# Drought Impact Visualisations

This notebook turns the results of **`drought_pipeline.ipynb`** into charts and maps.

The notebook can be followed top to bottom:

1. **Block 1 (Settings)** — Input the pipeline's `output/` folder(s). 
2. **Block 2 (Load and process the data)** — Reads everything and builds
   ready-to-use tables. Always run this before any figure. It does not need to be edited.
3. **Every cell after that is one figure.** Each one starts with a 
   **PARAMETERS** section (the year range, which languages, which impact types, which
   weather indicators, etc.). Change those few lines and re-run just that cell to get a
   different version of the same figure. The rest of each cell is the plotting code and does
   not need to be changed.

## What this notebook needs

**From `drought_pipeline.ipynb`** (just point Block 1 at the `output/` folder(s) from drought_pipeline.ipynb):

| File | Used for |
|---|---|
| `{LANG}_04_geoparsed.csv` | the filtering-funnel figure (optional) |
| `{LANG}_05_filtered.csv` | publication dates, headlines, sections |
| `{LANG}_09_impact_predictions.csv` | which impact types were found, and where |
| `{LANG}_10_locations_raw.csv` | every individual place mention (optional, for the weather-region maps) |
| `{LANG}_11_location_clusters.csv` | the one-main-place-per-article results (optional, for the canton maps) |

If the pipeline was run more than once into different `output/` folders (for example
once in `"train"` mode on the historic corpus, then again in `"apply"` mode on a new
batch of articles into `output_2025/`), every folder needs to be listed in Block 1 and this
notebook combines them automatically.

**Reference data that does *not* come from the pipeline** is needed, and Block 1 explains where
to get each one. If not available, every figure that needs one skips itself with a clear message
if it is missing, so the rest of the notebook can still be run):

- A Swiss canton map (a shapefile — a tile-grid cartogram is used here so all cantons
  are equally readable regardless of their real size)
- A weekly count of *all* Swissdox articles per language used as a baseline for reporting frequency
- Regional monthly drought/weather indicators and the
  matching region shapefile for the weather-correlation figures

---
# Block 1 — Settings

**This is the main cell that needs to be edited.** Everything below reads these values.

In [ ]:
# ============================================================================
# BLOCK 1 — SETTINGS
# ============================================================================

# ----------------------------------------------------------------------------
# 1. WHICH LANGUAGES, AND WHERE IS drought_pipeline.ipynb's OUTPUT?
# ----------------------------------------------------------------------------

# Which languages to include. Must match the LANGUAGE values used in the pipeline
# (a subset of "DE", "FR", "IT").
LANGUAGES = ["DE", "FR", "IT"]

# The OUTPUT_DIR(s) set in Block 1 of drought_pipeline.ipynb. Usually just one folder. 
# If the pipeline was run more than once into different folders 
# (for example once in "train" mode on the historic corpus, 
# and again in "apply" mode into a separate folder for each new batch of articles), 
# every folder should be listed here and they will all be combined automatically. 
# Order does not matter.
#
# Example with several batches:
#   OUTPUT_DIRS = ["output", "output_2025", "output_2026"]
OUTPUT_DIRS = ["output"]

# Where this notebook saves the figures it produces. Created automatically.
FIGURES_DIR = "figures"


# ----------------------------------------------------------------------------
# 2. REFERENCE DATA THAT DOES NOT COME FROM THE PIPELINE
# ----------------------------------------------------------------------------
# These are optional.

REFERENCE_DATA = {

    # A weekly count of ALL Swissdox articles (not just drought ones) per language.
    # Used to tell "the newspaper published more that year" apart from "drought
    # reporting specifically grew". Needs the columns: week_start, de_count, fr_count,
    # it_count.
    "swissdox_baseline": "reference_data/swissdox_baseline/weekly_counts.csv",

    # A Swiss canton shapefile, used for every canton-level map. A "tile-grid" style
    # map is used (every canton drawn as an equally-sized square), so that small cantons are as easy to read
    # as large ones. Needs a "Name" column with the canton's 2-letter abbreviation (e.g. "ZH", "VD") and a geometry column.
    "canton_tilemap": "reference_data/tilemap/Switzerland_Tiles_CH1903LV03.shp",
    "canton_tilemap_crs_epsg": 21781,   # the CRS the shapefile's coordinates are in

    # A shapefile of Swiss "Grossregionen" (large climate/hydrology regions), used only
    # for the weather-correlation figures in section 7. 
    # Needs a "Name" column matching the region names in REGION_MAP below.
    "bafu_regions": "reference_data/bafu_regions/BAFU_Grossregionen_erweitert_mitDoubs.shp",

    # Monthly regional drought/weather indicators, 
    # used only for the weather-correlation figures in section 7. 
    # A semicolon-separated CSV with 3 header rows to skip, a "measured_at" date column, a "drought_region_id" column,
    # and one column per indicator (see DROUGHT_VARIABLES below).
    "meteo_indicators": "reference_data/meteo_variables/monthly_historic_regions.csv",
}


# ----------------------------------------------------------------------------
# 3. STYLE — languages, impact types, fonts and colours used throughout
# ----------------------------------------------------------------------------
# Change these once here and every figure below picks them up automatically.

LANGUAGE_NAMES = {"DE": "German", "FR": "French", "IT": "Italian"}

# Colour-blind-safe language palette (ColorBrewer "Set2")
LANGUAGE_COLORS = {"DE": "#CE6500", "FR": "#8581B5", "IT": "#51AD91"}

# Typeface used in every matplotlib figure.
FONT_FAMILY = "sans-serif"
FONT_STACK = ["Arial", "Helvetica", "DejaVu Sans"]
TEXT_COLOR = "#404040"

# The same typeface/colour (for Plotly)
PLOTLY_FONT = dict(family="Arial, Helvetica, sans-serif", size=12, color=TEXT_COLOR)

# ----------------------------------------------------------------------------
# Impact-type grouping
# ----------------------------------------------------------------------------
# This combines the sub-impact types into their larger impact types (matching is case-insensitive).

IMPACT_GROUP_KEYWORDS = {
    "Agriculture":             ["agriculture"],
    "Public water supply":     ["supply", "publicwater"],
    "Aquatic ecosystems":      ["aquatic"],
    "Terrestrial ecosystems":  ["terrestrial"],
    "Energy":                  ["energy"],
    "Inland navigation":       ["navigation"],
}

# Display order and colour (Paul Tol "bright", colour-blind-safe) for the groups above.
IMPACT_GROUP_COLORS = {
    "Agriculture":            "#CCBB44",
    "Public water supply":    "#EE6677",
    "Aquatic ecosystems":     "#66CCEE",
    "Terrestrial ecosystems": "#228833",
    "Energy":                 "#BBBBBB",
    "Inland navigation":      "#AA3377",
}
IMPACT_GROUP_ORDER = list(IMPACT_GROUP_COLORS.keys())

import os
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.sans-serif": FONT_STACK,
    "text.color": TEXT_COLOR,
    "axes.labelcolor": TEXT_COLOR,
    "axes.titlecolor": TEXT_COLOR,
    "axes.edgecolor": "#BFBFBF",
    "xtick.color": TEXT_COLOR,
    "ytick.color": TEXT_COLOR,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 11,
    "savefig.dpi": 300,
})

os.makedirs(FIGURES_DIR, exist_ok=True)

print("Languages:  ", LANGUAGES)
print("Output dirs:", OUTPUT_DIRS)
print("Figures go to:", FIGURES_DIR)

---
# Block 2 — Load and process the data

**This needs to be run every time the notebook is opened.** 
Nothing below this point re-does any modelling or geoparsing,
it only loads the CSV files `drought_pipeline.ipynb` already produced and reshapes them
into a few tables that every figure below re-uses:

| Table | One row per... | Comes from |
|---|---|---|
| `articles` | article | `{LANG}_05_filtered.csv` |
| `filtering_raw` | article, *before* filtering (optional) | `{LANG}_04_geoparsed.csv` |
| `sentences` | impact-bearing sentence | `{LANG}_09_impact_predictions.csv` (+ dates from `articles`) |
| `article_impacts` | article x broad impact type | `sentences`, exploded and grouped |
| `locations` | main location per article (optional) | `{LANG}_11_location_clusters.csv` (+ dates from `articles`) |
| `best_location` | article (optional) | `locations`, keeping only the most salient row per article |
| `raw_mentions` | individual place mention (optional) | `{LANG}_10_locations_raw.csv` (+ dates from `articles`) |

Tables marked "optional" are only built if the matching file exists for at least one
language; the figures that need them check for this and skip themselves with a message
if the table is empty, rather than raising an error.

In [ ]:
# ============================================================================
# BLOCK 2 — LOAD AND PROCESS THE DATA
# ============================================================================

import ast
import glob
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ----------------------------------------------------------------------------
# 1. SMALL HELPERS
# ----------------------------------------------------------------------------

def find_stage_files(lang, stage):
    """Every file matching {output_dir}/{lang}_{stage}.csv across OUTPUT_DIRS."""
    paths = []
    for output_dir in OUTPUT_DIRS:
        path = os.path.join(output_dir, f"{lang}_{stage}.csv")
        if os.path.exists(path):
            paths.append(path)
    return paths


def load_stage_all(stage, languages=None, id_col=None, required=True):
    """
    Load one pipeline output stage (e.g. "09_impact_predictions") for every language
    in `languages`, across every folder in OUTPUT_DIRS, and concatenate the result with
    a "language" column added.

    id_col: the column that uniquely identifies an article/row in this stage (e.g.
    "file" or "content_id"). If given, rows are de-duplicated on (language, id_col),
    keeping the first occurrence.
    """
    languages = languages or LANGUAGES
    frames = []
    missing_languages = []
    for lang in languages:
        paths = find_stage_files(lang, stage)
        if not paths:
            missing_languages.append(lang)
            continue
        for path in paths:
            df = pd.read_csv(path, low_memory=False)
            df["language"] = lang
            frames.append(df)

    if missing_languages:
        checked = [os.path.join(d, f"{lang}_{stage}.csv") for d in OUTPUT_DIRS for lang in missing_languages]
        print(f"  WARNING: no '{stage}' file found for {missing_languages}. Checked: {checked}. "
              f"These languages will be MISSING from every figure until this is fixed — double-check "
              f"LANGUAGES and OUTPUT_DIRS in Block 1 match where drought_pipeline.ipynb actually wrote "
              f"its output.")

    if not frames:
        msg = f"No '{stage}' files found for {languages} in {OUTPUT_DIRS}."
        if required:
            raise FileNotFoundError(msg)
        print(f"  note: {msg} Figures needing this table will be skipped.")
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True)
    if id_col and id_col in out.columns:
        before = len(out)
        out = out.drop_duplicates(subset=["language", id_col], keep="first")
        if len(out) < before:
            print(f"  note: dropped {before - len(out):,} duplicate '{stage}' rows "
                  f"(same language + {id_col} seen in more than one OUTPUT_DIR)")

    counts_str = ", ".join(f"{lang}={n:,}" for lang, n in out["language"].value_counts().sort_index().items())
    print(f"  rows per language — {counts_str}")
    return out


def parse_label_list(value):
    """Read a Python-list-shaped CSV cell (e.g. "['agriculture', 'energy']") back into
    an actual list, tolerating empty/missing values."""
    if isinstance(value, list):
        return value
    if pd.isna(value) or str(value).strip() in ("", "[]"):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        return [str(v).strip() for v in parsed] if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


def merge_energy_labels(labels):
    """Treat energy_hydro, energy_nuclear etc. as one 'energy' category, matching how
    the pipeline's impact classifier (Block 8) was trained."""
    return sorted({("energy" if str(l).lower().startswith("energy") else l) for l in labels})


def group_impact(label):
    """Map one detailed impact label to a broad category using IMPACT_GROUP_KEYWORDS
    (Block 1). Returns None if no keyword matches."""
    label_lower = str(label).lower()
    for group, keywords in IMPACT_GROUP_KEYWORDS.items():
        if any(kw in label_lower for kw in keywords):
            return group
    return None


# Canton name (in German, French, Italian or English, as GeoNames/the geoparser might resolve it) 
CANTON_ABBR = {
    "zürich": "ZH", "zuerich": "ZH", "zurich": "ZH", "zurigo": "ZH",
    "bern": "BE", "berne": "BE", "berna": "BE",
    "luzern": "LU", "lucerne": "LU", "lucerna": "LU",
    "uri": "UR",
    "schwyz": "SZ", "svitto": "SZ",
    "obwalden": "OW", "obwald": "OW", "obvaldo": "OW",
    "nidwalden": "NW", "nidwald": "NW", "nidvaldo": "NW",
    "glarus": "GL", "glaris": "GL", "glarona": "GL",
    "zug": "ZG", "zoug": "ZG", "zugo": "ZG",
    "fribourg": "FR", "freiburg": "FR", "friburgo": "FR",
    "solothurn": "SO", "soleure": "SO", "soletta": "SO",
    "basel-city": "BS", "basel stadt": "BS", "bâle-ville": "BS", "basilea città": "BS",
    "basel-landschaft": "BL", "basel landschaft": "BL", "baselland": "BL", "basel-land": "BL",
    "bâle-campagne": "BL", "basilea campagna": "BL",
    "schaffhausen": "SH", "schaffhouse": "SH", "sciaffusa": "SH",
    "appenzell ausserrhoden": "AR", "appenzell außerrhoden": "AR",
    "appenzell rhodes-extérieures": "AR", "appenzello esterno": "AR",
    "appenzell innerrhoden": "AI", "appenzell rhodes-intérieures": "AI", "appenzello interno": "AI",
    "st. gallen": "SG", "st gallen": "SG", "sankt gallen": "SG", "saint gallen": "SG", "san gallo": "SG",
    "graubünden": "GR", "graubuenden": "GR", "grisons": "GR", "grigioni": "GR",
    "aargau": "AG", "argovie": "AG", "argovia": "AG",
    "thurgau": "TG", "thurgovie": "TG", "turgovia": "TG",
    "ticino": "TI", "tessin": "TI",
    "vaud": "VD", "waadt": "VD",
    "valais": "VS", "wallis": "VS", "vallese": "VS",
    "neuchâtel": "NE", "neuchatel": "NE", "neuenburg": "NE",
    "genève": "GE", "geneve": "GE", "genf": "GE", "geneva": "GE", "ginevra": "GE",
    "jura": "JU",
}
_KNOWN_ABBRS = set(CANTON_ABBR.values())


def to_canton_abbr(name):
    """Conversion of a canton name to its 2-letter abbreviation.
    Returns None if the name isn't recognised."""
    key = str(name).strip().lower()
    if key in CANTON_ABBR:
        return CANTON_ABBR[key]
    if key.upper() in _KNOWN_ABBRS:
        return key.upper()
    return None


def add_year_month(df, date_col="pubtime"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    if getattr(df[date_col].dt, "tz", None) is not None:
        df[date_col] = df[date_col].dt.tz_localize(None)
    df["year"] = df[date_col].dt.year
    df["month"] = df[date_col].dt.month
    df["year_month"] = df[date_col].dt.to_period("M").dt.to_timestamp()
    return df


# ----------------------------------------------------------------------------
# 2. ARTICLE METADATA  ({LANG}_05_filtered.csv)
# ----------------------------------------------------------------------------

print("Loading article metadata (05_filtered)...")
articles = load_stage_all("05_filtered", id_col="content_id")
articles = articles.rename(columns={"content_id": "file"})
articles["file"] = articles["file"].astype(str).str.strip()
articles = add_year_month(articles, "pubtime")
print(f"  {len(articles):,} articles across {articles['language'].nunique()} language(s)")

ARTICLE_META_COLS = ["file", "language", "pubtime", "year", "month", "year_month",
                      "head", "subhead", "rubric"]
ARTICLE_META_COLS = [c for c in ARTICLE_META_COLS if c in articles.columns]


# ----------------------------------------------------------------------------
# 3. FILTERING FUNNEL  ({LANG}_04_geoparsed.csv)
# ----------------------------------------------------------------------------

print("Loading pre-filtering snapshot (04_geoparsed, optional)...")
filtering_raw = load_stage_all("04_geoparsed", id_col="content_id", required=False)
if not filtering_raw.empty:
    filtering_raw = filtering_raw.rename(columns={"content_id": "file"})

# ----------------------------------------------------------------------------
# 3b. ARTICLE-RELEVANCE DECISIONS  ({LANG}_06_article_relevance.csv) — optional
# ----------------------------------------------------------------------------
# One row per article that survived Block 3's rule-based filters, carrying the
# article-relevance SVM's verdict in `is_relevant_article` ("yes"/"no").

print("Loading article-relevance decisions (06_article_relevance, optional)...")
_ar_frames = []
for _lang in LANGUAGES:
    _paths = find_stage_files(_lang, "06_article_relevance")
    if not _paths:
        print(f"  WARNING: no 06_article_relevance file found for {_lang} in {OUTPUT_DIRS}.")
        continue
    for _path in _paths:
        _header = pd.read_csv(_path, nrows=0).columns.tolist()
        if "is_relevant_article" not in _header:
            print(f"  WARNING: {_path} has no 'is_relevant_article' column "
                  f"(columns: {_header[:8]}). Skipping — was Block 5b run for {_lang}?")
            continue
        try:
            _ar = pd.read_csv(_path, usecols=["content_id", "is_relevant_article"])
        except Exception as _e:
            print(f"  WARNING: {_path} failed to parse ({type(_e).__name__}: {_e}). "
                  f"Retrying with the Python engine...")
            _ar = pd.read_csv(_path, usecols=["content_id", "is_relevant_article"],
                              engine="python", on_bad_lines="warn")
        _ar["language"] = _lang
        print(f"  {_lang}: read {len(_ar):,} rows from {_path}")
        _ar_frames.append(_ar)

if _ar_frames:
    article_relevance_raw = pd.concat(_ar_frames, ignore_index=True)
    assert "content_id" in article_relevance_raw.columns, \
        f"expected a 'content_id' column, got {article_relevance_raw.columns.tolist()}"
    _before = len(article_relevance_raw)
    article_relevance_raw = (
        article_relevance_raw
        .drop_duplicates(subset=["language", "content_id"], keep="first")
        .rename(columns={"content_id": "file"})
    )
    if len(article_relevance_raw) < _before:
        print(f"  note: dropped {_before - len(article_relevance_raw):,} duplicate row(s) "
              f"(same language + content_id).")
    _counts = ", ".join(f"{l}={n:,}" for l, n in
                        article_relevance_raw["language"].value_counts().sort_index().items())
    print(f"  rows per language — {_counts}")
    _missing = [l for l in LANGUAGES if l not in set(article_relevance_raw["language"])]
    if _missing:
        print(f"  WARNING: {_missing} produced no rows in the finished table. The funnel "
              f"figure will flag them rather than draw them as fully filtered out.")
else:
    print("  note: no '06_article_relevance' files found. The filtering-funnel figure will "
          "show only the three rule-based filters.")
    article_relevance_raw = pd.DataFrame()
del _ar_frames

# ----------------------------------------------------------------------------
# 4. IMPACT-BEARING SENTENCES  ({LANG}_09_impact_predictions.csv)
# ----------------------------------------------------------------------------
# One row per sentence the pipeline flagged as describing a drought impact, 
# with the model's predicted labels (pred_labels), the human-annotated labels where available
# (matched_labels — empty for un-annotated articles), and a prob_<label> confidence column per impact type the model was trained on.

print("Loading impact predictions (09_impact_predictions)...")
sentences = load_stage_all("09_impact_predictions", id_col=None, required=True)
sentences["file"] = sentences["file"].astype(str).str.strip()

for col in ("pred_labels", "matched_labels"):
    if col in sentences.columns:
        sentences[col] = sentences[col].apply(parse_label_list)
    else:
        sentences[col] = [[] for _ in range(len(sentences))]

# The ground-truth column keeps the pipeline's detailed annotation tags (e.g.
# "energy_hydro"), but the model's own predictions were trained on the merged "energy" category. 
sentences["matched_labels_merged"] = sentences["matched_labels"].apply(merge_energy_labels)

# Every "prob_<label>" column found
PROB_LABEL_COLUMNS = sorted({c for c in sentences.columns if c.startswith("prob_")})
IMPACT_LABELS_ALL = sorted({c[len("prob_"):] for c in PROB_LABEL_COLUMNS})

# Bring in the publication date / headline / section from `articles`.
sentences = sentences.merge(
    articles[ARTICLE_META_COLS], on=["file", "language"], how="left"
)
print(f"  {len(sentences):,} impact-bearing sentences, "
      f"{len(IMPACT_LABELS_ALL)} impact type(s): {IMPACT_LABELS_ALL}")


# ----------------------------------------------------------------------------
# 5. ARTICLE x BROAD IMPACT TYPE  (derived from `sentences`)
# ----------------------------------------------------------------------------
# Many figures report on 6 broad categories (Block 1's IMPACT_GROUP_KEYWORDS) rather than the detailed labels, 
# and at the ARTICLE level rather than per sentence (an article counts once per impact type, no matter how many sentences mention it).

_exploded = sentences[["file", "language", "year", "month", "year_month", "pred_labels"]].explode("pred_labels")
_exploded = _exploded.dropna(subset=["pred_labels"])
_exploded["impact_group"] = _exploded["pred_labels"].apply(group_impact)
article_impacts = (
    _exploded.dropna(subset=["impact_group"])
    [["file", "language", "year", "month", "year_month", "impact_group"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
del _exploded
print(f"  {len(article_impacts):,} article x broad-impact-type rows")


# ----------------------------------------------------------------------------
# 6. MAIN LOCATIONS  ({LANG}_11_location_clusters.csv)
# ----------------------------------------------------------------------------
# Already clustered and salience-filtered by the pipeline (Block 10b) — one row per
# main location per article.

print("Loading main locations (11_location_clusters, optional)...")
locations = load_stage_all("11_location_clusters", id_col=None, required=False)
if not locations.empty:
    locations["file"] = locations["file"].astype(str).str.strip()
    locations["canton_abbr"] = locations["canton"].apply(to_canton_abbr)

    unmapped = sorted(locations.loc[locations["canton_abbr"].isna() & locations["canton"].notna(),
                                     "canton"].unique())
    if unmapped:
        print(f"  note: canton name(s) not recognised (add them to CANTON_ABBR above "
              f"to include them in canton-level maps): {unmapped}")

    locations = locations.merge(articles[ARTICLE_META_COLS], on=["file", "language"], how="left")

    # The most salient location per article — used by any figure that needs exactly
    # one place per article (e.g. "which canton is this article mainly about").
    best_location = (
        locations.sort_values(["language", "file", "salience"], ascending=[True, True, False])
        .drop_duplicates(subset=["language", "file"], keep="first")
        .reset_index(drop=True)
    )
    print(f"  {len(locations):,} main-location rows, {len(best_location):,} articles with a location")
else:
    best_location = pd.DataFrame()


# ----------------------------------------------------------------------------
# 7. RAW PLACE MENTIONS  ({LANG}_10_locations_raw.csv)
# ----------------------------------------------------------------------------
# Every individual place mention before clustering, with its own latitude/longitude.
# Only used by the weather-region figures in section 7, 
# which need to spatially join each mention to a "Grossregion" rather than a canton.

print("Loading raw place mentions (10_locations_raw, optional)...")
raw_mentions = load_stage_all("10_locations_raw", id_col=None, required=False)
if not raw_mentions.empty:
    raw_mentions["file"] = raw_mentions["file"].astype(str).str.strip()
    if "has_swiss_toponym" in raw_mentions.columns:
        raw_mentions = raw_mentions[raw_mentions["has_swiss_toponym"] == 1].copy()
    raw_mentions = add_year_month(raw_mentions, "pubtime")
    print(f"  {len(raw_mentions):,} raw place mentions")


# ----------------------------------------------------------------------------
# 8. SIMPLE LOADERS FOR MAP GEOMETRY (only used by the canton-map / weather figures)
# ----------------------------------------------------------------------------
# These only run when a figure that needs them calls them (not here) 
# so Block 2 stays fast and doesn't require geopandas/the shapefiles unless those figures are actually used.

def load_canton_tilemap():
    """The canton tile-grid map from REFERENCE_DATA['canton_tilemap'], as a
    GeoDataFrame with columns 'canton_abbr' and 'geometry'. Returns None (and prints
    a note) if the file isn't there."""
    path = REFERENCE_DATA.get("canton_tilemap")
    if not path or not os.path.exists(path):
        print(f"  note: canton tilemap not found at '{path}' (see Block 1) — skipping.")
        return None
    import geopandas as gpd
    epsg = REFERENCE_DATA.get("canton_tilemap_crs_epsg", 21781)
    cantons = gpd.read_file(path).set_crs(epsg=epsg, allow_override=True).to_crs(epsg=4326)
    cantons = cantons[["Name", "geometry"]].rename(columns={"Name": "canton_abbr"})
    # Drop any lake/water polygons
    cantons = cantons[~cantons["canton_abbr"].str.contains("Bodensee|Lac L", case=False, na=False)]
    return cantons.reset_index(drop=True)


def load_bafu_regions():
    """The BAFU 'Grossregionen' shapefile from REFERENCE_DATA['bafu_regions'], used
    by the weather-correlation figures. Returns None if not available."""
    path = REFERENCE_DATA.get("bafu_regions")
    if not path or not os.path.exists(path):
        print(f"  note: BAFU regions shapefile not found at '{path}' (see Block 1) — skipping.")
        return None
    import geopandas as gpd
    regions = gpd.read_file(path)


    def try_fix(name):
        try:
            return str(name).encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return name

    original_names = regions["Name"]
    fixed_names = original_names.apply(try_fix)
    n_match_original = original_names.isin(REGION_MAP).sum()
    n_match_fixed = fixed_names.isin(REGION_MAP).sum()

    if n_match_fixed > n_match_original:
        regions["Name"] = fixed_names
        print(f"  note: applied a text-encoding fix to region names ({n_match_fixed}/{len(regions)} "
              f"matched REGION_MAP afterwards, vs {n_match_original}/{len(regions)} before).")
    else:
        print(f"  note: region names matched REGION_MAP as-is ({n_match_original}/{len(regions)}).")

    n_matched = max(n_match_original, n_match_fixed)
    if n_matched < len(regions):
        unmatched = sorted(set(regions["Name"]) - set(REGION_MAP))
        print(f"  WARNING: {len(regions) - n_matched}/{len(regions)} region name(s) did not match "
              f"REGION_MAP and will show as blank/grey in the weather-correlation maps: {unmatched}. "
              f"Add them to REGION_MAP (Figure 13's parameters) if you recognise them.")

    return regions.to_crs("EPSG:4326")


print()
print("=" * 70)
print("SUMMARY — rows per language in each table (0 usually means that language's file")
print("wasn't found — check the WARNING messages above, and LANGUAGES/OUTPUT_DIRS in Block 1)")
print("=" * 70)
summary_tables = {
    "articles": articles, "filtering_raw": filtering_raw, "sentences": sentences,
    "article_impacts": article_impacts, "locations": locations,
    "best_location": best_location, "raw_mentions": raw_mentions,
}
summary_rows = {}
for table_name, table_df in summary_tables.items():
    if table_df.empty or "language" not in table_df.columns:
        summary_rows[table_name] = {lang: 0 for lang in LANGUAGES}
    else:
        summary_rows[table_name] = table_df["language"].value_counts().reindex(LANGUAGES, fill_value=0).to_dict()
print(pd.DataFrame(summary_rows).T[LANGUAGES])
print("=" * 70)

print()
print("=" * 70)
print("BLOCK 2 DONE — tables ready: articles, filtering_raw, sentences, article_impacts,")
print("                              locations, best_location, raw_mentions")
print("=" * 70)

---
## Figure 1 — Swissdox database size per language, over time

How many articles (of any topic, not just drought) Swissdox holds per language and
year. This is **not** pipeline output. Swissdox does not publically expose this as a file, so the
numbers below were entered by hand from the Swissdox web interface. Update the
`SWISSDOX_SIZE` table below to update this figure with newer numbers.

In [ ]:
# ============================================================================
# FIGURE 1 — PARAMETERS
# ============================================================================

# One row per year. Update these numbers by hand for newer year. Not derived from pipeline.
SWISSDOX_SIZE = {
    "Year": list(range(1990, 2025)),
    "German": [3549, 4029, 5130, 20296, 90109, 115899, 161214, 161500, 239029, 341271,
               448464, 462981, 530019, 551901, 561120, 575197, 650793, 758850, 848198,
               904824, 844242, 811704, 821049, 865527, 861258, 840266, 897711, 898229,
               1049305, 1063300, 1279782, 1382820, 1331746, 1153612, 1135833],
    "French": [33, 223, 559, 2021, 6129, 6323, 15915, 48862, 65272, 73300, 78465, 83660,
               81185, 34863, 97517, 123350, 134898, 117512, 126394, 122600, 129910,
               141582, 176688, 178610, 175831, 194104, 253910, 276931, 257021, 249985,
               213941, 223370, 229202, 224634, 226002],
    "Italian": [0, 0, 0, 4, 9, 30, 37, 39, 14, 0, 1, 2, 0, 2, 1, 0, 0, 2, 0, 818, 732,
                848, 700, 582, 727, 562, 880, 7705, 1047, 1014, 4797, 17551, 18815,
                18259, 22006],
}

# ============================================================================
# FIGURE 1 — PLOT
# ============================================================================
import matplotlib.pyplot as plt

swissdox_df = pd.DataFrame(SWISSDOX_SIZE)
lang_display_colors = {"German": "#CE6500", "French": "#8581B5", "Italian": "#51AD91"}

plt.figure(figsize=(11, 6.5))
for lang in ["German", "French", "Italian"]:
    plt.plot(swissdox_df["Year"], swissdox_df[lang], label=lang,
              color=lang_display_colors[lang], linewidth=3, solid_capstyle="round")

plt.title("Swissdox database size per language", fontsize=16, color=TEXT_COLOR, pad=20)
plt.xlabel("Year", fontsize=12, color=TEXT_COLOR, labelpad=10)
plt.ylabel("Number of articles", fontsize=12, color=TEXT_COLOR, labelpad=10)
plt.xticks(swissdox_df["Year"], rotation=90, fontsize=10, color=TEXT_COLOR)

ax = plt.gca()
ax.get_yaxis().set_major_formatter(
    plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x)).replace(",", "'"))
)
plt.grid(True, axis="y", color="#D9D9D9", linewidth=1)
plt.xlim(swissdox_df["Year"].min(), swissdox_df["Year"].max())
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#BFBFBF")
ax.tick_params(axis="both", which="both", length=0)
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "01_swissdox_size.png"), dpi=300, bbox_inches="tight")
plt.show()

---
## Figure 2 — Share of articles removed by each filtering step, per language

Reports Block 3f of the pipeline as a chart: of everything read
in, what share was dropped as a duplicate, by section, or as foreign reporting, and what
share survived. Needs `{LANG}_04_geoparsed.csv` (Block 2 loads it as `filtering_raw`).

In [ ]:
# ============================================================================
# FIGURE 2 — PARAMETERS
# ============================================================================

FIG2_LANGUAGES = LANGUAGES     # which languages to show, e.g. ["DE", "FR"]

# ============================================================================
# FIGURE 2 — PLOT
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

if filtering_raw.empty:
    print("Skipping: no '04_geoparsed' file found for any language (see Block 2).")
else:
    steps = ["Duplicate removal", "Section removal", "Foreign-reporting removal", "Remaining corpus"]
    step_colors = ["#a6cee3", "#1f78b4", "#b2df8a", "#fb9a99"]

    rows = []
    for lang in FIG2_LANGUAGES:
        sub = filtering_raw[filtering_raw["language"] == lang]
        if sub.empty:
            continue
        total = len(sub)
        originals = sub[sub["duplicate"].str.lower() == "original"]
        dup_removed = total - len(originals)

        rubric_col = originals["rubric_relevance"].str.lower()
        rubric_removed = (rubric_col == "irrelevant").sum()
        reached_swiss_filter = originals[rubric_col == "relevant"]

        if "swiss" in reached_swiss_filter.columns:
            foreign_removed = (reached_swiss_filter["swiss"].str.lower() == "no").sum()
        else:
            foreign_removed = 0

        kept = (sub["keep_final"] == 1).sum() if "keep_final" in sub.columns else (
            total - dup_removed - rubric_removed - foreign_removed
        )

        rows.append({
            "language": lang,
            "Duplicate removal": dup_removed,
            "Section removal": rubric_removed,
            "Foreign-reporting removal": foreign_removed,
            "Remaining corpus": kept,
            "total": total,
        })

    funnel_df = pd.DataFrame(rows).set_index("language")
    funnel_pct = funnel_df[steps].div(funnel_df["total"], axis=0) * 100

    y_pos = np.arange(len(funnel_pct))[::-1] * 1.6
    bar_height = 0.55
    fig, ax = plt.subplots(figsize=(10, 1.6 * len(funnel_pct) + 2))

    left = np.zeros(len(funnel_pct))
    for step, color in zip(steps, step_colors):
        values = funnel_pct[step].values
        ax.barh(y_pos, values, left=left, height=bar_height, label=step,
                color=color, edgecolor="white", linewidth=0.6)
        for y, v, l in zip(y_pos, values, left):
            if v >= 4:
                ax.text(l + v / 2, y, f"{v:.0f}%", va="center", ha="center",
                        fontsize=9, color="#333333")
        left += values

    ax.set_title("Share of articles by filtering step, per language", fontsize=13,
                 fontweight="bold", pad=20, color="#333333")
    ax.set_xlabel("Share of articles", fontsize=11, labelpad=10)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([LANGUAGE_NAMES.get(l, l) for l in funnel_pct.index], fontsize=12)
    ax.set_ylim(y_pos.min() - 0.9, y_pos.max() + 0.9)
    ax.set_xlim(0, 100)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
    ax.grid(axis="x", color="#E0E0E0", alpha=0.7, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_color("#CCCCCC")
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False, fontsize=9.5)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "02_filtering_funnel.png"), bbox_inches="tight")
    plt.show()

---
## Figure 3 — Predicted vs. annotated impact types, per language

For every annotated sentence, which impact type did a human annotate that the model did
*not* predict, and the other way around? Uses the `matched_labels` (human annotation) and
`pred_labels` (model prediction) columns from `sentences`, so it only covers sentences from articles annotated.

In [ ]:
# ============================================================================
# FIGURE 3 — PARAMETERS
# ============================================================================

FIG3_LANGUAGES = LANGUAGES

FIG3_LABEL_DISPLAY_NAMES = {
    "agriculture_annual": "Agriculture (annual)",
    "agriculture_general": "Agriculture (general)",
    "agriculture_livestock": "Agriculture (livestock)",
    "agriculture_permanent": "Agriculture (permanent)",
    "ecosystems_aquatic": "Aquatic ecosystems",
    "ecosystems_terrestrial": "Terrestrial ecosystems",
    "energy": "Energy",
    "inlandnavigation": "Inland navigation",
    "publicwatersupply": "Public water supply",
}

# ============================================================================
# FIGURE 3 — COMPUTE
# ============================================================================
import itertools
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap
import seaborn as sns

def strict_confusions(df):
    """For each pair of labels, how often did the model predict `pred` when the true
    label was `true` (and `pred` was not itself also correct on that sentence)?"""
    df = df[df["matched_labels_merged"].map(len) > 0]
    all_labels = sorted(
        set(itertools.chain.from_iterable(df["matched_labels_merged"])) |
        set(itertools.chain.from_iterable(df["pred_labels"]))
    )
    mat = pd.DataFrame(0, index=all_labels, columns=all_labels, dtype=int)
    for _, row in df.iterrows():
        true_set, pred_set = set(row["matched_labels_merged"]), set(row["pred_labels"])
        for true_label in true_set:
            if true_label in pred_set:
                continue
            for pred_label in pred_set:
                if pred_label != true_label:
                    mat.loc[true_label, pred_label] += 1
    for label in all_labels:
        mat.loc[label, label] = 0
    return mat

matrices, all_labels_global = {}, set()
for lang in FIG3_LANGUAGES:
    mat = strict_confusions(sentences[sentences["language"] == lang])
    matrices[lang] = mat
    all_labels_global.update(mat.index)
    all_labels_global.update(mat.columns)

if not matrices or not all_labels_global:
    print("Skipping: no annotated sentences (non-empty matched_labels) found.")
else:
    all_labels_global = sorted(all_labels_global)
    formatted_labels = [FIG3_LABEL_DISPLAY_NAMES.get(l, l) for l in all_labels_global]
    for lang in matrices:
        matrices[lang] = matrices[lang].reindex(index=all_labels_global, columns=all_labels_global, fill_value=0)
    vmax = max(1, max(m.values.max() for m in matrices.values()))

    lang_cmaps = {l: LinearSegmentedColormap.from_list(f"{l}_cmap", ["#ffffff", LANGUAGE_COLORS.get(l, "#666666")])
                  for l in matrices}

    fig, axes = plt.subplots(1, len(matrices), figsize=(4 * len(matrices) + 2, 6.6), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for i, (ax, (lang, mat)) in enumerate(zip(axes, matrices.items())):
        cmap = lang_cmaps[lang].copy()
        cmap.set_under("white")
        bounds = np.arange(0, vmax + 1, 1)
        norm = BoundaryNorm(bounds, cmap.N)
        sns.heatmap(mat, ax=ax, cmap=cmap, linewidths=0.5, norm=norm, square=True,
            cbar=True,
            cbar_kws={"orientation": "horizontal", "pad": 0.03, "shrink": 0.8,
                      "label": "Number of sentences"})
        ax.set_title(LANGUAGE_NAMES.get(lang, lang), fontsize=15)
        ax.set_xlabel("Predicted label", fontsize=12)
        if i == 0:
            ax.set_ylabel("True label", fontsize=12)
            ax.set_yticklabels(formatted_labels, rotation=0, fontsize=10)
        else:
            ax.set_ylabel("")
            ax.set_yticklabels([])
            ax.tick_params(axis="y", left=False)
        ax.set_xticklabels(formatted_labels, rotation=90, fontsize=10)
    fig.suptitle("Impact types confused by the multi-label classifier", fontsize=17, fontweight="bold")
    plt.savefig(os.path.join(FIGURES_DIR, "03_confusion_matrices.png"), dpi=300, bbox_inches="tight")
    plt.show()

---
## Figure 4 — Yearly counts of drought-impact sentences, raw and normalised

Top panel: how many impact-bearing sentences were found per year, per language. 
Bottom panel: the same, divided by the *total* number of Swissdox articles published that year
(so growth in reporting overall doesn't look like growth in drought reporting specifically). 
The bottom panel needs `REFERENCE_DATA["swissdox_baseline"]` (Block 1).

In [ ]:
# ============================================================================
# FIGURE 4 — PARAMETERS
# ============================================================================

FIG4_LANGUAGES = LANGUAGES
FIG4_START_YEAR = 1990
FIG4_END_YEAR = 2024

# What to count per year: "sentences" (every impact-bearing sentence, matching the
# original figure) or "articles" (each article counted once, no matter how many impact
# sentences it contains).
FIG4_COUNT_UNIT = "articles"     # "sentences" or "articles"

# ============================================================================
# FIGURE 4 — PLOT
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

sub = sentences[
    sentences["language"].isin(FIG4_LANGUAGES) &
    sentences["year"].between(FIG4_START_YEAR, FIG4_END_YEAR)
]

if FIG4_COUNT_UNIT == "articles":
    counts = sub.drop_duplicates(["language", "file"]).groupby(["year", "language"]).size()
else:
    counts = sub.groupby(["year", "language"]).size()

media_df = counts.unstack("language").reindex(columns=FIG4_LANGUAGES).fillna(0)

baseline_path = REFERENCE_DATA.get("swissdox_baseline")
have_baseline = baseline_path and os.path.exists(baseline_path)

if have_baseline:
    baseline = pd.read_csv(baseline_path)
    baseline["week_start"] = pd.to_datetime(baseline["week_start"], errors="coerce")
    baseline["year"] = baseline["week_start"].dt.year
    baseline_yearly = baseline.groupby("year")[["de_count", "fr_count", "it_count"]].sum()
    baseline_yearly = baseline_yearly.rename(columns={"de_count": "DE", "fr_count": "FR", "it_count": "IT"})

    norm_df = pd.DataFrame(index=media_df.index)
    for lang in FIG4_LANGUAGES:
        if lang in baseline_yearly.columns:
            norm_df[lang] = (media_df[lang] / baseline_yearly[lang].replace(0, np.nan)) * 100
    n_panels = 2
else:
    print("note: swissdox_baseline not found (see Block 1) — showing raw counts only.")
    n_panels = 1

years = media_df.index
x = np.arange(len(years))
width = 0.8 / max(len(FIG4_LANGUAGES), 1)

fig, axes = plt.subplots(n_panels, 1, figsize=(10, 4 * n_panels), sharex=True)
axes = np.atleast_1d(axes)

ax1 = axes[0]
for i, lang in enumerate(FIG4_LANGUAGES):
    ax1.bar(x + i * width, media_df[lang], width=width, label=LANGUAGE_NAMES.get(lang, lang),
            color=LANGUAGE_COLORS.get(lang))
ax1.set_ylabel("Number of articles", fontsize=10)
ax1.set_title("Drought articles per year, raw counts", fontsize=12, fontweight="bold")
ax1.set_xticks(x + width * (len(FIG4_LANGUAGES) - 1) / 2)
ax1.set_xticklabels([int(y) for y in years], rotation=90, fontsize=9)
ax1.grid(axis="y", alpha=0.4)
ax1.legend(fontsize=9, loc="upper left")

if have_baseline:
    ax2 = axes[1]
    for i, lang in enumerate(FIG4_LANGUAGES):
        if lang in norm_df.columns:
            ax2.bar(x + i * width, norm_df[lang], width=width, label=LANGUAGE_NAMES.get(lang, lang),
                    color=LANGUAGE_COLORS.get(lang))
    ax2.set_ylabel("Share of language corpus (%)", fontsize=10)
    ax2.set_title("Drought articles per year, as % of all Swissdox articles in that language", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Year", fontsize=10)
    ax2.grid(axis="y", alpha=0.4)
    ax2.legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "04_yearly_counts.png"), bbox_inches="tight")
plt.show()

---
## Figure 5 — Average monthly distribution of impact sentences, per language

Of everything found for a given language, what share appears in each calendar month?
(All years collapsed together)

In [ ]:
# ============================================================================
# FIGURE 5 — PARAMETERS
# ============================================================================

FIG5_LANGUAGES = LANGUAGES
FIG5_START_YEAR = 1990
FIG5_END_YEAR = 2025

# ============================================================================
# FIGURE 5 — PLOT
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MONTH_ABBR_BY_NUMBER = {i + 1: m for i, m in enumerate(MONTH_ORDER)}
MONTH_COLORS = ["#2b6cb0", "#135294", "#4aba7f", "#38a169", "#2f855a", "#f6e05e",
                 "#ecc94b", "#deb628", "#eb7e57", "#dd6b20", "#bf540d", "#4e8dcf"]

sub = sentences[
    sentences["language"].isin(FIG5_LANGUAGES) &
    sentences["year"].between(FIG5_START_YEAR, FIG5_END_YEAR)
].dropna(subset=["pubtime"]).copy()

sub["month_abbr"] = pd.to_datetime(sub["pubtime"]).dt.month.map(MONTH_ABBR_BY_NUMBER)

monthly_pct = {}
for lang in FIG5_LANGUAGES:
    counts = sub.loc[sub["language"] == lang, "month_abbr"].value_counts()
    monthly_pct[lang] = (counts / counts.sum() * 100).reindex(MONTH_ORDER).fillna(0)
monthly_df = pd.DataFrame(monthly_pct)

x = np.arange(len(FIG5_LANGUAGES)) * 0.8
width = 0.06
fig, ax = plt.subplots(figsize=(14, 6))

for i, month in enumerate(MONTH_ORDER):
    ax.bar(x + i * width, monthly_df.loc[month], width=width, color=MONTH_COLORS[i])

tick_positions, tick_labels = [], []
for i, month in enumerate(MONTH_ORDER):
    tick_positions.extend(x + i * width)
    tick_labels.extend([month] * len(FIG5_LANGUAGES))
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=90)

group_center = width * (len(MONTH_ORDER) - 1) / 2
for i, lang in enumerate(FIG5_LANGUAGES):
    ax.text(x[i] + group_center, -2.5, LANGUAGE_NAMES.get(lang, lang), ha="center", va="top",
            fontsize=11, fontweight="bold")

ax.set_ylabel("Percentage of sentences (%)")
ax.set_title("Average monthly distribution of impact sentences, by language")
ax.set_axisbelow(True)
ax.grid(axis="x", linestyle="--", linewidth=0.4, alpha=0.35)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "05_seasonal_distribution.png"), bbox_inches="tight")
plt.show()

---
## Figure 6 — Yearly impact-type composition across Switzerland

For each year, what share of articles is dominated by each broad impact type (the type
with the most sentences in that article), stacked to 100%, with the total number of
articles overlaid as a line. Restricted to articles that also have a resolved location,
so the result lines up with the canton-level maps later in the notebook.

In [ ]:
# ============================================================================
# FIGURE 6 — PARAMETERS
# ============================================================================

FIG6_LANGUAGES = LANGUAGES
FIG6_START_YEAR = 1995
FIG6_END_YEAR = 2025


FIG6_IMPACT_GROUPS = IMPACT_GROUP_ORDER

# Only count articles that also have a resolved main location (recommended as it keeps this
# figure consistent with the canton-level maps). 
# Set to False to include every article regardless of whether a location was found.
FIG6_REQUIRE_LOCATION = True

# ============================================================================
# FIGURE 6 — PLOT
# ============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def build_yearly_composition(languages, start_year, end_year, impact_groups, require_location):
    df = article_impacts[
        article_impacts["language"].isin(languages) &
        article_impacts["impact_group"].isin(impact_groups) &
        article_impacts["year"].between(start_year, end_year)
    ].copy()

    if require_location and not best_location.empty:
        located = best_location[["language", "file"]].drop_duplicates()
        df = df.merge(located, on=["language", "file"], how="inner")

    # dominant impact type per article: the group(s) with the most sentences
    counts = df.groupby(["language", "file", "year", "impact_group"]).size().reset_index(name="n")
    counts["max_n"] = counts.groupby(["language", "file"])["n"].transform("max")
    dominant = counts[counts["n"] == counts["max_n"]]

    yearly = dominant.groupby(["year", "impact_group"]).size().reset_index(name="article_count")
    grid = pd.MultiIndex.from_product(
        [sorted(dominant["year"].dropna().unique()), impact_groups], names=["year", "impact_group"]
    ).to_frame(index=False)
    yearly = grid.merge(yearly, on=["year", "impact_group"], how="left")
    yearly["article_count"] = yearly["article_count"].fillna(0).astype(int)
    yearly["year_total"] = yearly.groupby("year")["article_count"].transform("sum")
    yearly = yearly[yearly["year_total"] > 0].copy()
    yearly["proportion"] = yearly["article_count"] / yearly["year_total"]
    return yearly

yearly_counts = build_yearly_composition(
    FIG6_LANGUAGES, FIG6_START_YEAR, FIG6_END_YEAR, FIG6_IMPACT_GROUPS, FIG6_REQUIRE_LOCATION
)

if yearly_counts.empty:
    print("Skipping: no data for the chosen parameters (check FIG6_REQUIRE_LOCATION if "
          "you have no location data).")
else:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for impact in FIG6_IMPACT_GROUPS:
        subset = yearly_counts[yearly_counts["impact_group"] == impact]
        fig.add_trace(go.Bar(
            x=subset["year"], y=subset["proportion"], name=impact,
            marker_color=IMPACT_GROUP_COLORS.get(impact, "#999999"),
            hovertemplate="Year=%{x}<br>Impact=%{fullData.name}<br>Proportion=%{y:.2%}<extra></extra>",
        ), secondary_y=False)

    totals = yearly_counts.groupby("year")["year_total"].max().reset_index()
    fig.add_trace(go.Scatter(
        x=totals["year"], y=totals["year_total"], name="Total articles",
        mode="lines+markers", legendrank=1, line=dict(color="black", width=2),
        marker=dict(size=6), hovertemplate="Year=%{x}<br>Total=%{y}<extra></extra>",
    ), secondary_y=True)

    fig.update_layout(
        barmode="stack", height=650, width=1200,
        title=dict(text="<b>Yearly impact-type composition</b>", x=0.5, xanchor="center",
                   y=0.9, yanchor="top", font=dict(size=20, color=TEXT_COLOR)),
        legend_title_text="Impact Type", margin=dict(t=90, r=50, l=60, b=40), font=PLOTLY_FONT,
    )
    fig.update_yaxes(title_text="Proportion of articles", tickformat=".0%", range=[0, 1], secondary_y=False)
    fig.update_yaxes(title_text="Total number of articles", secondary_y=True)
    
    years = sorted(yearly_counts["year"].unique())
    fig.update_xaxes(
        dtick=1,
        tickangle=60,
        range=[min(years) - 0.5, max(years) + 0.5]
    )

    fig.show()
    try:
        fig.write_image(os.path.join(FIGURES_DIR, "06_yearly_impact_composition.png"), scale=3)
    except Exception as e:
        print(f"note: could not save a PNG copy ({e}). The figure above is still interactive; "
              f"'pip install -U kaleido' enables saving Plotly figures to file.")

---
## Figure 7 — Monthly impact-type composition across Switzerland

The same as Figure 6, but collapsing all years together by calendar month (all
Januaries combined, all Februaries combined, ...) to show seasonality in *which
kind* of impact gets reported, not just how much reporting there is.

In [ ]:
# ============================================================================
# FIGURE 7 — PARAMETERS
# ============================================================================

FIG7_LANGUAGES = LANGUAGES
FIG7_START_YEAR = 1994
FIG7_END_YEAR = 2025
FIG7_IMPACT_GROUPS = IMPACT_GROUP_ORDER
FIG7_REQUIRE_LOCATION = True

# ============================================================================
# FIGURE 7 — PLOT
# ============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MONTH_LABELS = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                 7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}

def build_monthly_composition(languages, start_year, end_year, impact_groups, require_location):
    df = article_impacts[
        article_impacts["language"].isin(languages) &
        article_impacts["impact_group"].isin(impact_groups) &
        article_impacts["year"].between(start_year, end_year)
    ].copy()

    if require_location and not best_location.empty:
        located = best_location[["language", "file"]].drop_duplicates()
        df = df.merge(located, on=["language", "file"], how="inner")

    counts = df.groupby(["language", "file", "month", "impact_group"]).size().reset_index(name="n")
    counts["max_n"] = counts.groupby(["language", "file"])["n"].transform("max")
    dominant = counts[counts["n"] == counts["max_n"]]

    monthly = dominant.groupby(["month", "impact_group"]).size().reset_index(name="article_count")
    grid = pd.MultiIndex.from_product([range(1, 13), impact_groups], names=["month", "impact_group"]).to_frame(index=False)
    monthly = grid.merge(monthly, on=["month", "impact_group"], how="left")
    monthly["article_count"] = monthly["article_count"].fillna(0).astype(int)
    monthly["month_total"] = monthly.groupby("month")["article_count"].transform("sum")
    monthly = monthly[monthly["month_total"] > 0].copy()
    monthly["proportion"] = monthly["article_count"] / monthly["month_total"]
    monthly["month_name"] = monthly["month"].map(MONTH_LABELS)
    return monthly

monthly_counts = build_monthly_composition(
    FIG7_LANGUAGES, FIG7_START_YEAR, FIG7_END_YEAR, FIG7_IMPACT_GROUPS, FIG7_REQUIRE_LOCATION
)

if monthly_counts.empty:
    print("Skipping: no data for the chosen parameters.")
else:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for impact in FIG7_IMPACT_GROUPS:
        subset = monthly_counts[monthly_counts["impact_group"] == impact]
        fig.add_trace(go.Bar(
            x=subset["month_name"], y=subset["proportion"], name=impact,
            marker_color=IMPACT_GROUP_COLORS.get(impact, "#999999"),
            hovertemplate="Month=%{x}<br>Impact=%{fullData.name}<br>Proportion=%{y:.2%}<extra></extra>",
        ), secondary_y=False)

    totals = monthly_counts.groupby("month")[["month_total", "month_name"]].first().reset_index()
    fig.add_trace(go.Scatter(
        x=totals["month_name"], y=totals["month_total"], name="Total articles",
        mode="lines+markers", legendrank=1, line=dict(color="black", width=2),
        marker=dict(size=6), hovertemplate="Month=%{x}<br>Total=%{y}<extra></extra>",
    ), secondary_y=True)

    fig.update_layout(
        barmode="stack", height=650, width=1200,
        title=dict(text="<b>Monthly impact-type composition</b>", x=0.5, xanchor="center",
                   y=0.9, yanchor="top", font=dict(size=20, color=TEXT_COLOR)),
        legend_title_text="Impact Type", margin=dict(t=90, r=50, l=60, b=40), font=PLOTLY_FONT,
    )
    fig.update_yaxes(title_text="Proportion of articles", tickformat=".0%", range=[0, 1], secondary_y=False)
    fig.update_yaxes(title_text="Total number of articles", secondary_y=True)
    fig.update_xaxes(categoryorder="array", categoryarray=list(MONTH_LABELS.values()), tickangle=60)

    fig.show()
    try:
        fig.write_image(os.path.join(FIGURES_DIR, "07_monthly_impact_composition.png"), scale=3)
    except Exception as e:
        print(f"note: could not save a PNG copy ({e}). The figure above is still interactive; "
              f"'pip install -U kaleido' enables saving Plotly figures to file.")

---
## Figure 8 — Deviation from each canton's average reporting, per year

For each canton, is a given year reported on more or less than that canton's own
long-term average? Shown as a standardised residual (a "chi" score): 
0 = exactly the long-term average, 
positive = more than usual, 
negative = less. 

Comparing a canton to *its own* average (rather than to other cantons) avoids the map just showing which
cantons are biggest. Needs `REFERENCE_DATA["canton_tilemap"]` (Block 1).

In [ ]:
# ============================================================================
# FIGURE 8 — PARAMETERS
# ============================================================================

FIG8_LANGUAGES = LANGUAGES
FIG8_START_YEAR = 1995
FIG8_END_YEAR = 2025
FIG8_N_COLS = 5                 # small maps per row
FIG8_CHI_VMIN, FIG8_CHI_VMAX = -15, 20
FIG8_CMAP_NAME = "RdBu_r"

# ============================================================================
# FIGURE 8 — PLOT
# ============================================================================
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

cantons_gdf = load_canton_tilemap()

if cantons_gdf is None or best_location.empty:
    print("Skipping: needs both the canton tilemap and location data (Block 2).")
else:
    sub = best_location[
        best_location["language"].isin(FIG8_LANGUAGES) &
        best_location["year"].between(FIG8_START_YEAR, FIG8_END_YEAR) &
        best_location["canton_abbr"].notna()
    ]

    if sub.empty:
        n_in_range = best_location[
            best_location["language"].isin(FIG8_LANGUAGES) &
            best_location["year"].between(FIG8_START_YEAR, FIG8_END_YEAR)
        ]
        n_with_canton = n_in_range["canton_abbr"].notna().sum()
        print(f"Skipping: no articles with a recognised canton in {FIG8_LANGUAGES}, "
              f"{FIG8_START_YEAR}-{FIG8_END_YEAR} ({len(n_in_range):,} articles in range, "
              f"{n_with_canton:,} with a canton Block 2 could match — see the 'canton name(s) "
              f"not recognised' note printed by Block 2, and extend CANTON_ABBR there if needed).")
    else:
        counts = sub.groupby(["year", "canton_abbr"]).size().reset_index(name="article_count")
        all_cantons = cantons_gdf["canton_abbr"].unique()
        all_years = sorted(counts["year"].unique())
        full_index = pd.MultiIndex.from_product([all_years, all_cantons], names=["year", "canton_abbr"])
        counts = counts.set_index(["year", "canton_abbr"]).reindex(full_index, fill_value=0).reset_index()

        canton_means = counts.groupby("canton_abbr")["article_count"].mean().rename("expected")
        counts = counts.merge(canton_means, on="canton_abbr")
        counts["chi"] = (counts["article_count"] - counts["expected"]) / np.sqrt(counts["expected"].replace(0, np.nan))
        counts["chi"] = counts["chi"].replace([np.inf, -np.inf], np.nan).fillna(0)

        n_years = len(all_years)
        n_rows = int(np.ceil(n_years / FIG8_N_COLS))
        fig, axes = plt.subplots(n_rows, FIG8_N_COLS, figsize=(12, 2.6 * n_rows))
        axes = np.atleast_2d(axes)
        norm = Normalize(vmin=FIG8_CHI_VMIN, vmax=FIG8_CHI_VMAX)
        cmap = plt.get_cmap(FIG8_CMAP_NAME)

        for idx, year in enumerate(all_years):
            row, col = divmod(idx, FIG8_N_COLS)
            ax = axes[row, col]
            year_df = counts.loc[counts["year"] == year, ["canton_abbr", "chi"]]
            merged = cantons_gdf.merge(year_df, on="canton_abbr", how="left")
            merged.plot(column="chi", cmap=cmap, norm=norm, ax=ax, edgecolor="#666666",
                        linewidth=0.4, missing_kwds={"color": "#eeeeee"})
            ax.set_title(str(int(year)), fontsize=11, fontweight="bold", pad=2)
            ax.set_aspect("equal")
            ax.axis("off")

        for idx in range(n_years, n_rows * FIG8_N_COLS):
            row, col = divmod(idx, FIG8_N_COLS)
            axes[row, col].axis("off")

        fig.suptitle("Deviation from each canton's average reporting", fontsize=18, fontweight="bold", y=0.98)
        fig.subplots_adjust(hspace=0.05, wspace=0.05, right=0.9)
        cbar_ax = fig.add_axes([0.92, 0.35, 0.02, 0.3])
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        fig.colorbar(sm, cax=cbar_ax, label="Chi residual")

        fig.savefig(os.path.join(FIGURES_DIR, "08_canton_reporting_deviation.png"), dpi=200, bbox_inches="tight")
        plt.show()

---
## Figure 9 — Which impact type dominates each canton, each year

One small map per year; each canton is coloured by whichever broad impact type made up
the largest share of that canton's articles that year. A canton is only given a colour
if it has at least `FIG9_N_MIN` classified articles that year and the winning type beats
the runner-up by at least `FIG9_MARGIN_MIN`; otherwise it is shown as "mixed / no clear
dominant". Needs `REFERENCE_DATA["canton_tilemap"]` (Block 1).

In [ ]:
# ============================================================================
# FIGURE 9 — PARAMETERS
# ============================================================================

FIG9_LANGUAGES = LANGUAGES
FIG9_START_YEAR = 1995
FIG9_END_YEAR = 2024
FIG9_IMPACT_GROUPS = IMPACT_GROUP_ORDER

FIG9_N_MIN = 1          # minimum classified articles in a canton-year to declare a winner
FIG9_MARGIN_MIN = 0.00  # winning type's article-share must exceed the runner-up by at least this

# Optional per-type handicap: weight < 1 holds a type to a HIGHER bar (it must
# out-accumulate rivals by 1/weight to win). Leave empty ({}) for no handicap.
FIG9_TYPE_HANDICAP = {"Terrestrial ecosystems": 0.85}

FIG9_N_COLS = 5

# ============================================================================
# FIGURE 9 — PLOT
# ============================================================================
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

cantons_gdf = load_canton_tilemap()

if cantons_gdf is None or best_location.empty:
    print("Skipping: needs both the canton tilemap and location data (Block 2).")
else:
    article_canton = best_location[
        best_location["language"].isin(FIG9_LANGUAGES) &
        best_location["year"].between(FIG9_START_YEAR, FIG9_END_YEAR) &
        best_location["canton_abbr"].notna()
    ][["language", "file", "year", "canton_abbr"]].rename(columns={"canton_abbr": "canton"})

    if article_canton.empty:
        print(f"Skipping: no articles with a recognised canton in {FIG9_LANGUAGES}, "
              f"{FIG9_START_YEAR}-{FIG9_END_YEAR} (see the note Block 2 prints about "
              f"unrecognised canton names, and extend CANTON_ABBR there if needed).")
    else:
        baseline_counts = article_canton.groupby(["year", "canton"]).size().reset_index(name="n_spatial")

        impacts = article_impacts[
            article_impacts["language"].isin(FIG9_LANGUAGES) &
            article_impacts["impact_group"].isin(FIG9_IMPACT_GROUPS)
        ].merge(article_canton[["language", "file", "canton"]], on=["language", "file"], how="inner")

        # Proportional vote: each article's impact types are normalised to shares summing
        # to 1 (so an article with many sentences doesn't dominate one with few), summed
        # per canton-year, then the winner is the type with the highest handicapped share.
        per_article = impacts.groupby(["language", "file", "year", "canton", "impact_group"]).size().rename("n").reset_index()
        per_article["share"] = per_article["n"] / per_article.groupby(["language", "file"])["n"].transform("sum")

        canton_year = per_article.groupby(["year", "canton", "impact_group"])["share"].sum().rename("score").reset_index()
        n_articles = per_article.groupby(["year", "canton"])["file"].nunique().rename("n_articles").reset_index()
        canton_year = canton_year.merge(n_articles, on=["year", "canton"])
        canton_year["dominance"] = canton_year["score"] / canton_year["n_articles"]
        canton_year["weight"] = canton_year["impact_group"].map(FIG9_TYPE_HANDICAP).fillna(1.0)
        canton_year["dom_adj"] = canton_year["dominance"] * canton_year["weight"]

        canton_year["rank"] = canton_year.groupby(["year", "canton"])["dom_adj"].rank(method="first", ascending=False)
        top = (canton_year[canton_year["rank"] == 1][["year", "canton", "impact_group", "dominance", "dom_adj", "n_articles"]]
               .rename(columns={"impact_group": "top_impact", "dominance": "dom1", "dom_adj": "adj1"}))
        second = canton_year[canton_year["rank"] == 2][["year", "canton", "dom_adj"]].rename(columns={"dom_adj": "adj2"})
        winners = top.merge(second, on=["year", "canton"], how="left")
        winners["adj2"] = winners["adj2"].fillna(0.0)
        winners["margin"] = winners["adj1"] - winners["adj2"]

        all_cantons = cantons_gdf["canton_abbr"].dropna().unique()
        all_years = list(range(FIG9_START_YEAR, FIG9_END_YEAR + 1))
        full = pd.MultiIndex.from_product([all_years, all_cantons], names=["year", "canton"]).to_frame(index=False)
        grid = full.merge(baseline_counts, on=["year", "canton"], how="left").merge(winners, on=["year", "canton"], how="left")
        grid["n_spatial"] = grid["n_spatial"].fillna(0).astype(int)
        grid["margin"] = grid["margin"].fillna(0.0)

        color_map = dict(IMPACT_GROUP_COLORS)
        color_map["Mixed / no clear dominant"] = "#888888"
        color_map["No reporting available"] = "#ffffff"
        display_order = FIG9_IMPACT_GROUPS + ["Mixed / no clear dominant", "No reporting available"]

        conditions = [grid["n_spatial"].lt(FIG9_N_MIN), grid["top_impact"].isna(), grid["margin"] < FIG9_MARGIN_MIN]
        choices = ["No reporting available", "Mixed / no clear dominant", "Mixed / no clear dominant"]
        grid["impact_display"] = np.select(conditions, choices, default=grid["top_impact"].astype("object"))

        n_years = len(all_years)
        n_rows = int(np.ceil(n_years / FIG9_N_COLS))
        fig, axes = plt.subplots(n_rows, FIG9_N_COLS, figsize=(12, 2.6 * n_rows))
        axes = np.atleast_2d(axes)

        for idx, year in enumerate(all_years):
            row, col = divmod(idx, FIG9_N_COLS)
            ax = axes[row, col]
            year_df = grid.loc[grid["year"] == year, ["canton", "impact_display"]].rename(columns={"canton": "canton_abbr"})
            merged = cantons_gdf.merge(year_df, on="canton_abbr", how="left")
            merged["color"] = merged["impact_display"].map(color_map).fillna("#eeeeee")
            merged.plot(color=list(merged["color"]), edgecolor="white", linewidth=0.4, ax=ax)
            ax.set_title(str(year), fontsize=11, fontweight="bold", pad=2)
            ax.set_aspect("equal")
            ax.axis("off")

        for idx in range(n_years, n_rows * FIG9_N_COLS):
            row, col = divmod(idx, FIG9_N_COLS)
            axes[row, col].axis("off")

        legend_elements = [Patch(facecolor=color_map[label], edgecolor="#999999", label=label) for label in display_order]
        fig.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, -0.03),
                   ncol=4, fontsize=9, frameon=False)
        fig.suptitle("Dominant impact type per canton, per year", fontsize=18, fontweight="bold", y=0.98)
        fig.subplots_adjust(hspace=0.05, wspace=0.05)

        fig.savefig(os.path.join(FIGURES_DIR, "09_dominant_impact_per_canton.png"), dpi=200, bbox_inches="tight")
        plt.show()

---
## Figure 10 — Top cantons reported on, per language

The cantons that appear most often as an article's main location, one ranking per
language. Computed directly from `locations` (Block 2).

In [ ]:
# ============================================================================
# FIGURE 10 — PARAMETERS
# ============================================================================

FIG10_LANGUAGES = LANGUAGES
FIG10_TOP_N = 10

# ============================================================================
# FIGURE 10 — PLOT
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

if locations.empty:
    print("Skipping: no location data (see Block 2).")
else:
    fig, axes = plt.subplots(1, len(FIG10_LANGUAGES), figsize=(4 * len(FIG10_LANGUAGES), 4), sharex=False)
    axes = np.atleast_1d(axes)
    fig.suptitle(f"Top {FIG10_TOP_N} cantons reported on, by language", fontsize=13, fontweight="bold", y=1.03)

    for i, (lang, ax) in enumerate(zip(FIG10_LANGUAGES, axes)):
        sub = locations[(locations["language"] == lang) & locations["canton_abbr"].notna()]
        counts = sub["canton_abbr"].value_counts().head(FIG10_TOP_N)
        pct = (counts / sub["canton_abbr"].value_counts().sum() * 100).reset_index()
        pct.columns = ["canton_abbr", "pct"]

        sns.barplot(x="pct", y="canton_abbr", data=pct, ax=ax, color=LANGUAGE_COLORS.get(lang))
        ax.set_title(f"{LANGUAGE_NAMES.get(lang, lang)} sources", fontsize=11, fontweight="bold")
        ax.set_xlabel("Percentage (%)", fontsize=9.5)
        ax.set_ylabel("Canton" if i == 0 else "")
        sns.despine(ax=ax)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "10_top_cantons_per_language.png"), dpi=200, bbox_inches="tight")
    plt.show()

---
## Figure 11 — Impact-type mix by language and "domestic vs. outside" coverage

For each language, what share of impact types is reported when the article's home
canton is talking about itself ("domestic") vs. reporting on another canton
("outside")? **This is not produced by the pipeline** — it needs you to already know
which canton each newspaper is based in, which the pipeline has no way of inferring —
so the counts below are a manually-filled-in table. Edit `FIG11_DATA` with your own
numbers (e.g. from a spreadsheet where you've tagged each newspaper's home canton) to
refresh this figure.

In [ ]:
# ============================================================================
# FIGURE 11 — PARAMETERS
# ============================================================================

FIG11_LANGUAGES = LANGUAGES
FIG11_START_YEAR = 1995
FIG11_END_YEAR = 2024

# Which cantons count as a language's own language region. An article is "domestic"
# when its main canton (Block 2's `best_location`) is one of its own language's
# cantons, and "outside" otherwise. Bilingual cantons can appear in more than one
# list (BE and VS for German and French, GR for German and Italian).
FIG11_HOME_CANTONS = {
    "DE": ["ZH", "BE", "LU", "UR", "SZ", "OW", "NW", "GL", "ZG", "SO", "BS", "BL",
           "SH", "AR", "AI", "SG", "GR", "AG", "TG", "VS", "FR"],
    "FR": ["GE", "VD", "NE", "JU", "FR", "VS", "BE"],
    "IT": ["TI", "GR"],
}

FIG11_SECTORS = IMPACT_GROUP_ORDER
FIG11_COLORS = [IMPACT_GROUP_COLORS[s] for s in FIG11_SECTORS]

FIG11_MIN_LABEL_PCT = 3    # don't print a number inside slices thinner than this
FIG11_MIN_ARTICLES = 20    # warn about rows built on fewer articles than this

# ============================================================================
# FIGURE 11 — PLOT
# ============================================================================
import matplotlib.pyplot as plt

if best_location.empty or article_impacts.empty:
    print("Skipping: needs location data (11_location_clusters) and impact data — see Block 2.")
else:
    # One row per article: its language and its main canton.
    scope = best_location[
        best_location["language"].isin(FIG11_LANGUAGES) &
        best_location["year"].between(FIG11_START_YEAR, FIG11_END_YEAR) &
        best_location["canton_abbr"].notna()
    ][["language", "file", "canton_abbr"]].drop_duplicates(subset=["language", "file"])

    home = {lang: {c.upper() for c in cantons} for lang, cantons in FIG11_HOME_CANTONS.items()}
    scope["coverage"] = [
        "domestic" if canton in home.get(lang, set()) else "outside"
        for lang, canton in zip(scope["language"], scope["canton_abbr"])
    ]

    # Article x broad impact type, restricted to those articles.
    impacts = (article_impacts[article_impacts["impact_group"].isin(FIG11_SECTORS)]
               .merge(scope, on=["language", "file"], how="inner"))

    if impacts.empty:
        print("Skipping: no articles with both a recognised canton and a classified impact type "
              f"in {FIG11_LANGUAGES}, {FIG11_START_YEAR}-{FIG11_END_YEAR}.")
    else:
        labels, shares_rows, row_counts = [], [], []
        for lang in reversed(FIG11_LANGUAGES):
            sub = impacts[impacts["language"] == lang]
            n_lang = sub["file"].nunique()
            for coverage in ("outside", "domestic"):
                part = sub[sub["coverage"] == coverage]
                n_articles = part["file"].nunique()
                counts = (part.groupby("impact_group")["file"].nunique()
                          .reindex(FIG11_SECTORS, fill_value=0).astype(float))
                total = counts.sum()
                shares = counts / total * 100 if total else counts
                pct_of_lang = (n_articles / n_lang * 100) if n_lang else 0.0
                labels.append(f"{LANGUAGE_NAMES.get(lang, lang)}: {coverage} ({pct_of_lang:.0f}%)")
                shares_rows.append(shares.values)
                row_counts.append(n_articles)

        df11 = pd.DataFrame(shares_rows, index=labels, columns=FIG11_SECTORS)

        print("Articles behind each row (bars are shares of that row's article x impact pairs):")
        print(pd.DataFrame({"n_articles": row_counts}, index=labels).to_string())
        thin = [lab for lab, n in zip(labels, row_counts) if 0 < n < FIG11_MIN_ARTICLES]
        if thin:
            print(f"  WARNING: fewer than {FIG11_MIN_ARTICLES} articles behind: {thin} — "
                  f"those percentages are noisy.")

        fig, ax = plt.subplots(figsize=(11, 6), dpi=150)
        y_positions = []
        for i in range(len(FIG11_LANGUAGES)):
            y_positions.extend([i * 2.5, i * 2.5 + 1])
        y_positions = y_positions[:len(labels)]
        left_base = np.zeros(len(labels))

        for sector, color in zip(FIG11_SECTORS, FIG11_COLORS):
            widths = df11[sector].values
            bars = ax.barh(y_positions, widths, left=left_base, height=0.7, label=sector,
                           color=color, edgecolor="none")
            for bar, width in zip(bars, widths):
                if width >= FIG11_MIN_LABEL_PCT:
                    ax.text(bar.get_x() + width / 2, bar.get_y() + bar.get_height() / 2,
                            f"{width:.0f}", va="center", ha="center", color="#222222", fontsize=11)
            left_base += widths

        ax.set_title("Impact distribution by language and domestic/outside coverage",
                     fontsize=13, fontweight="bold", pad=20, color="#333333")
        ax.set_xlabel("share of articles", fontsize=11, labelpad=10)
        ax.set_yticks(y_positions)
        ax.set_yticklabels(labels, fontsize=11, color="#555555")
        for label in ax.get_yticklabels():
            label.set_horizontalalignment("left")
        ax.tick_params(axis="y", pad=130)
        ax.set_ylim(min(y_positions) - 0.6, max(y_positions) + 0.6)
        ax.set_xlim(0, 100)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
        ax.grid(axis="x", color="#E0E0E0", alpha=0.7, zorder=0)
        ax.set_axisbelow(True)
        for spine in ["top", "right", "left", "bottom"]:
            ax.spines[spine].set_color("#CCCCCC")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=len(FIG11_SECTORS),
                  frameon=False, fontsize=10)

        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, "11_domestic_vs_outside.png"), bbox_inches="tight")
        plt.show()

---
## Figure 12 — Which impact types get reported together, per language

For each pair of impact types, the Jaccard similarity: of all articles mentioning
*either* type, what share mention *both*? 
1.0 = always reported together, 0.0 = never.

In [ ]:
# ============================================================================
# FIGURE 12 — PARAMETERS
# ============================================================================

FIG12_LANGUAGES = LANGUAGES
FIG12_IMPACT_GROUPS = IMPACT_GROUP_ORDER

# ============================================================================
# FIGURE 12 — PLOT
# ============================================================================
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

def jaccard_matrix(languages_subset, impact_groups):
    sub = article_impacts[
        article_impacts["language"] == languages_subset
    ]
    binary = pd.crosstab(sub["file"], sub["impact_group"]).reindex(columns=impact_groups, fill_value=0)
    binary = (binary > 0).astype(int)
    mat = pd.DataFrame(index=impact_groups, columns=impact_groups, dtype=float)
    for i in impact_groups:
        for j in impact_groups:
            a, b = binary[i], binary[j]
            union = ((a == 1) | (b == 1)).sum()
            mat.loc[i, j] = ((a == 1) & (b == 1)).sum() / union if union > 0 else np.nan
    return mat

matrices = {lang: jaccard_matrix(lang, FIG12_IMPACT_GROUPS) for lang in FIG12_LANGUAGES}
lang_cmaps = {l: LinearSegmentedColormap.from_list(f"{l}_cmap", ["#ffffff", LANGUAGE_COLORS.get(l, "#666666")])
              for l in FIG12_LANGUAGES}

fig, axes = plt.subplots(1, len(FIG12_LANGUAGES), figsize=(4.2 * len(FIG12_LANGUAGES), 4.5), constrained_layout=True)
axes = np.atleast_1d(axes)

for i, (ax, lang) in enumerate(zip(axes, FIG12_LANGUAGES)):
    sns.heatmap(matrices[lang], ax=ax, cmap=lang_cmaps[lang], linewidths=0.5, vmin=0, vmax=1,
                square=True, cbar=True, annot=True, fmt=".2f",
                cbar_kws={"orientation": "horizontal", "pad": 0.02, "shrink": 0.8})
    ax.set_title(LANGUAGE_NAMES.get(lang, lang), fontsize=15, fontweight="bold")
    ax.set_xlabel("Impact type", fontsize=12)
    if i == 0:
        ax.set_ylabel("Impact type", fontsize=12)
    else:
        ax.set_ylabel("")
        ax.tick_params(axis="y", left=False)
    ax.tick_params(axis="x", rotation=90)
    ax.tick_params(axis="y", rotation=0)

fig.suptitle("Co-occurrence of impact types (Jaccard similarity)", fontsize=18, fontweight="bold")
fig.savefig(os.path.join(FIGURES_DIR, "12_impact_cooccurrence.png"), dpi=300, bbox_inches="tight")
plt.show()

---
## Section 7 — Media coverage vs. regional weather/drought indicators

The next five cells compare drought reporting to actual regional weather data. **This cell must be run first**.
It assigns every place mention to a "Grossregion" and correlates each of the 18 available indicators against media
coverage. 
Figures 14–17 plot what this cell computes. 
Needs `REFERENCE_DATA["bafu_regions"]`, `REFERENCE_DATA["meteo_indicators"]` and
`REFERENCE_DATA["swissdox_baseline"]` (Block 1), plus `raw_mentions` (Block 2). 
If any of these are missing, this cell and Figures 14–17 print a note and are skipped.

In [ ]:
# ============================================================================
# FIGURE 13 — PARAMETERS
# ============================================================================

FIG13_LANGUAGES = LANGUAGES
FIG13_START_YEAR = 2000
FIG13_END_YEAR = 2024
FIG13_WINSORIZE_TOP = 0.02   # cap the top X% of normalised-media outliers before correlating
FIG13_MIN_OBS = 24           # a region/variable needs at least this many months of overlap

# The first year of coverage to trust for each language (e.g. if Italian reporting
# barely exists before 2015, exclude those years from ITS correlations specifically).
FIG13_LANG_START_YEAR = {"DE": 1990, "FR": 1990, "IT": 2015}

# All 18 indicators available in the reference meteo file.
FIG13_DROUGHT_VARS = [
    "cdi", "precip_1m_index", "precip_3m_index", "precip_24m_index", "hydro_index",
    "soil_moisture_index", "precip_sum_1m", "precip_sum_3m", "precip_sum_6m",
    "precip_sum_12m", "precip_sum_24m", "spi_1m", "spi_3m", "spi_6m", "spi_12m",
    "spi_24m", "soil_moisture_ufc", "vhi",
]

FIG13_DROUGHT_VAR_LABELS = {
    "cdi": "CDI", "precip_1m_index": "Precip. 1m index", "precip_3m_index": "Precip. 3m index",
    "precip_24m_index": "Precip. 24m index", "hydro_index": "Hydro index",
    "soil_moisture_index": "Soil moisture index", "precip_sum_1m": "Precip. sum 1m",
    "precip_sum_3m": "Precip. sum 3m", "precip_sum_6m": "Precip. sum 6m",
    "precip_sum_12m": "Precip. sum 12m", "precip_sum_24m": "Precip. sum 24m",
    "spi_1m": "SPI-1", "spi_3m": "SPI-3", "spi_6m": "SPI-6", "spi_12m": "SPI-12",
    "spi_24m": "SPI-24", "soil_moisture_ufc": "Soil moisture UFC", "vhi": "VHI",
}

# Region name (as it appears in the BAFU shapefile's "Name" column) -> numeric ID used
# by the meteo indicator file's "drought_region_id" column.
REGION_MAP = {
    "östlicher Jura": 31, "Balsthal": 32, "Unteres Emmental": 33, "Berner Mittelland": 34,
    "Westliches Berner Oberland": 35, "Greyerzerland": 36, "Oberaargau": 37,
    "Oberes Emmental": 38, "Freiamt": 39, "westliche Zentralschweiz": 40,
    "Östliches Berner Oberland": 41, "östliches Mittelland": 42, "Zürichseebecken": 43,
    "östliche Zentralschweiz": 44, "Urner Alpen": 45, "Surselva": 46, "Alpstein": 47,
    "Glarner Alpen": 48, "nördliches Mittelbünden": 49, "südliches Mittelbünden": 50,
    "Rheintal": 51, "Prättigau": 52, "Freiberge": 53, "westlicher Jura": 54,
    "westliches Mittelland": 55, "Genferseebecken": 56, "Chablais": 57,
    "Walliser Südalpen": 58, "nördliches Wallis": 59, "Vispertäler": 60,
    "Aletschgebiet": 61, "Westliches Tessin": 62, "Nordtessin": 63, "Östliches Tessin": 64,
    "Sottoceneri": 65, "Bündner Südtäler": 66, "Oberengadin": 67, "Unterengadin": 68,
}
ID_TO_REGION = {v: k for k, v in REGION_MAP.items()}

# ============================================================================
# FIGURE 13 — PROCESSING
# ============================================================================
import re
import warnings
from scipy.stats import pearsonr, spearmanr
from scipy.stats.mstats import winsorize

warnings.filterwarnings("ignore", category=RuntimeWarning)

FIG13_READY = True
meteo_path = REFERENCE_DATA.get("meteo_indicators")
baseline_path = REFERENCE_DATA.get("swissdox_baseline")

if raw_mentions.empty:
    print("Skipping section 7: no raw place mentions (10_locations_raw) loaded — see Block 2.")
    FIG13_READY = False
regions_gdf = load_bafu_regions() if FIG13_READY else None
if FIG13_READY and regions_gdf is None:
    FIG13_READY = False
if FIG13_READY and not (meteo_path and os.path.exists(meteo_path)):
    print(f"Skipping section 7: meteo indicators file not found at '{meteo_path}' (see Block 1).")
    FIG13_READY = False
if FIG13_READY and not (baseline_path and os.path.exists(baseline_path)):
    print(f"Skipping section 7: swissdox baseline not found at '{baseline_path}' (see Block 1).")
    FIG13_READY = False

if FIG13_READY:
    import geopandas as gpd

    def safe_divide(numerator, denominator):
        return np.where((denominator.notna()) & (denominator != 0), numerator / denominator, np.nan)

    def adjust_drought_direction(series, var_name):
        """Re-orient every indicator so that a HIGHER value always means MORE drought."""
        var_name = var_name.lower()
        if var_name in ("cdi", "precip_1m_index", "precip_3m_index", "precip_24m_index",
                         "hydro_index", "soil_moisture_index"):
            return series - 1
        return -series   # spi_*, precip_sum_*, soil_moisture_ufc, vhi

    def winsorize_series(series, top=0.0):
        if series.dropna().empty:
            return series
        return pd.Series(winsorize(series, limits=[0, top]), index=series.index)

    # --- monthly regional drought indicators ---
    drought_monthly_raw = pd.read_csv(meteo_path, skiprows=3, sep=";")
    drought_monthly_raw["measured_at"] = pd.to_datetime(drought_monthly_raw["measured_at"], format="%d.%m.%Y")
    drought_monthly_raw["year"] = drought_monthly_raw["measured_at"].dt.year
    drought_monthly_raw["month"] = drought_monthly_raw["measured_at"].dt.month
    drought_monthly_raw["month_start"] = pd.to_datetime(
        dict(year=drought_monthly_raw["year"], month=drought_monthly_raw["month"], day=1))
    drought_monthly = (
        drought_monthly_raw.groupby(["drought_region_id", "year", "month", "month_start"], as_index=False)
        .mean(numeric_only=True)
    )

    # --- swissdox baseline, monthly, per language ---
    baseline = pd.read_csv(baseline_path)
    baseline["week_start"] = pd.to_datetime(baseline["week_start"])
    baseline["year"] = baseline["week_start"].dt.year
    baseline["month"] = baseline["week_start"].dt.month
    baseline_monthly = baseline.groupby(["year", "month"], as_index=False).sum(numeric_only=True)
    baseline_monthly["month_start"] = pd.to_datetime(
        dict(year=baseline_monthly["year"], month=baseline_monthly["month"], day=1))

    # --- media mentions -> region, monthly, per language (from raw_mentions) ---
    def load_media_language(lang):
        data = raw_mentions[raw_mentions["language"] == lang].dropna(subset=["latitude", "longitude"])
        gdf = gpd.GeoDataFrame(data, geometry=gpd.points_from_xy(data["longitude"], data["latitude"]), crs="EPSG:4326")
        gdf = gpd.sjoin(gdf, regions_gdf[["Name", "geometry"]], how="inner", predicate="within")
        gdf["drought_region_id"] = gdf["Name"].map(REGION_MAP)
        out = (gdf.groupby(["drought_region_id", "year", "month", "year_month"], as_index=False)["file"]
               .nunique().rename(columns={"file": "Media", "year_month": "month_start"}))
        return out

    media_by_lang = {lang: load_media_language(lang) for lang in FIG13_LANGUAGES}

    def media_all_languages(langs):
        combined = pd.concat([media_by_lang[l] for l in langs], ignore_index=True)
        return combined.groupby(["drought_region_id", "year", "month", "month_start"], as_index=False)["Media"].sum()

    def baseline_combined_for(langs):
        cols = [f"{l.lower()}_count" for l in langs]
        out = baseline_monthly[["year", "month", "month_start"] + [c for c in cols if c in baseline_monthly.columns]].copy()
        out["baseline_total"] = out[[c for c in cols if c in baseline_monthly.columns]].sum(axis=1)
        return out

    def run_full_variable_comparison():
        media_monthly = media_all_languages(FIG13_LANGUAGES)
        baseline_combined = baseline_combined_for(FIG13_LANGUAGES)
        region_ids = sorted(drought_monthly["drought_region_id"].dropna().unique())
        rows = []
        for drought_var in FIG13_DROUGHT_VARS:
            if drought_var not in drought_monthly.columns:
                continue
            for region_nr in region_ids:
                d = drought_monthly.loc[drought_monthly["drought_region_id"] == region_nr,
                                          ["year", "month", "month_start", drought_var]].rename(columns={drought_var: "Drought"})
                d["Drought_adj"] = adjust_drought_direction(d["Drought"], drought_var)
                d = d[(d["year"] >= FIG13_START_YEAR) & (d["year"] <= FIG13_END_YEAR)]
                m = media_monthly.loc[media_monthly["drought_region_id"] == region_nr, ["year", "month", "month_start", "Media"]]
                df = d.merge(m, on=["year", "month", "month_start"], how="left")
                df["Media"] = df["Media"].fillna(0)
                df = df.merge(baseline_combined[["year", "month", "month_start", "baseline_total"]],
                              on=["year", "month", "month_start"], how="left")
                df["Media_norm"] = safe_divide(df["Media"], df["baseline_total"])
                df["Media_wins"] = winsorize_series(df["Media_norm"], top=FIG13_WINSORIZE_TOP)
                valid = df[["Drought_adj", "Media_wins"]].dropna()
                if len(valid) < FIG13_MIN_OBS or valid["Drought_adj"].nunique() <= 1 or valid["Media_wins"].nunique() <= 1:
                    continue
                r, _ = pearsonr(valid["Drought_adj"], valid["Media_wins"])
                rho, _ = spearmanr(valid["Drought_adj"], valid["Media_wins"])
                rows.append(dict(region_nr=region_nr, region_name=ID_TO_REGION.get(region_nr), drought_var=drought_var,
                                  var_label=FIG13_DROUGHT_VAR_LABELS.get(drought_var, drought_var),
                                  pearson_r=r, spearman_rho=rho, n=len(valid)))
        region_corrs = pd.DataFrame(rows)
        summary = (region_corrs.groupby(["drought_var", "var_label"])
                   .agg(mean_pearson_r=("pearson_r", "mean"), n_regions=("region_nr", "nunique"))
                   .reset_index().sort_values("mean_pearson_r", ascending=False).reset_index(drop=True))
        return region_corrs, summary

    def compute_language_split_comparison():
        all_summaries = {}
        for lang in FIG13_LANGUAGES:
            lang_start = max(FIG13_START_YEAR, FIG13_LANG_START_YEAR.get(lang, FIG13_START_YEAR))
            media_monthly = media_by_lang[lang]
            baseline_col = f"{lang.lower()}_count"
            if baseline_col not in baseline_monthly.columns:
                continue
            baseline_lang = baseline_monthly[["year", "month", "month_start", baseline_col]]
            region_ids = sorted(drought_monthly["drought_region_id"].dropna().unique())
            rows = []
            for drought_var in FIG13_DROUGHT_VARS:
                if drought_var not in drought_monthly.columns:
                    continue
                for region_nr in region_ids:
                    d = drought_monthly.loc[drought_monthly["drought_region_id"] == region_nr,
                                              ["year", "month", "month_start", drought_var]].rename(columns={drought_var: "Drought"})
                    d["Drought_adj"] = adjust_drought_direction(d["Drought"], drought_var)
                    d = d[(d["year"] >= lang_start) & (d["year"] <= FIG13_END_YEAR)]
                    m = media_monthly.loc[media_monthly["drought_region_id"] == region_nr, ["year", "month", "month_start", "Media"]]
                    df = d.merge(m, on=["year", "month", "month_start"], how="left")
                    df["Media"] = df["Media"].fillna(0)
                    df = df.merge(baseline_lang, on=["year", "month", "month_start"], how="left")
                    df["Media_norm"] = safe_divide(df["Media"], df[baseline_col])
                    df["Media_wins"] = winsorize_series(df["Media_norm"])
                    valid = df[["Drought_adj", "Media_wins"]].dropna()
                    if len(valid) < FIG13_MIN_OBS or valid["Drought_adj"].nunique() <= 1 or valid["Media_wins"].nunique() <= 1:
                        continue
                    r, _ = pearsonr(valid["Drought_adj"], valid["Media_wins"])
                    rows.append(dict(region_nr=region_nr, drought_var=drought_var,
                                      var_label=FIG13_DROUGHT_VAR_LABELS.get(drought_var, drought_var),
                                      pearson_r=r, n=len(valid)))
            rc = pd.DataFrame(rows)
            if rc.empty:
                continue
            all_summaries[lang] = (rc.groupby(["drought_var", "var_label"])
                                    .agg(mean_r=("pearson_r", "mean"), n_regions=("region_nr", "nunique"))
                                    .reset_index().sort_values("mean_r", ascending=False).reset_index(drop=True))

        records = [
            {"var_label": row["var_label"], "language": lang, "mean_r": row["mean_r"], "n_regions": row["n_regions"]}
            for lang, summ in all_summaries.items() for _, row in summ.iterrows()
        ]
        return all_summaries, pd.DataFrame(records)

    print("Running region x variable correlation scan (this can take a little while)...")
    region_corrs, summary = run_full_variable_comparison()
    all_summaries, combined = compute_language_split_comparison()
    print(f"Done. region_corrs: {region_corrs.shape}, summary: {summary.shape}, "
          f"languages compared: {list(all_summaries.keys())}")

---
## Figure 14 — Which weather indicator best explains media coverage, per region

Each region coloured by whichever of the 18 indicators has the highest mean Pearson
correlation with (normalised) media coverage in that region. Needs Figure 13 to have
run successfully first.

In [ ]:
# ============================================================================
# FIGURE 14 — PARAMETERS
# ============================================================================

FIG14_METRIC = "pearson_r"
FIG14_TITLE = "Best-correlating weather indicator per region"

# Explicit colours for certain indicators, anything else falls back to a colour-blind-safe qualitative palette automatically.
FIG14_FIXED_VAR_COLORS = {
    "CDI": "#fb9a99", "Hydro index": "#a6cee3", "Precip. 3m index": "#1f78b4",
    "Soil moisture UFC": "#33a02c", "Soil moisture index": "#b2df8a",
}

# ============================================================================
# FIGURE 14 — PLOT
# ============================================================================
if not FIG13_READY:
    print("Skipping: run Figure 13's cell successfully first.")
else:
    import textwrap
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import matplotlib.patheffects as pe
    from matplotlib.patches import Patch

    mean_r = region_corrs.groupby(["region_nr", "var_label"])[FIG14_METRIC].mean().reset_index()
    best = mean_r.sort_values(FIG14_METRIC, ascending=False).drop_duplicates("region_nr").rename(columns={FIG14_METRIC: "best_r"})

    gdf = regions_gdf.copy()
    gdf["region_nr"] = gdf["Name"].map(REGION_MAP)
    gdf = gdf.merge(best[["region_nr", "var_label", "best_r"]], on="region_nr", how="left")

    n_colored = gdf["var_label"].notna().sum()
    print(f"{n_colored}/{len(gdf)} regions have a best-correlating indicator to show "
          f"(the rest will be grey — usually because their region name didn't match REGION_MAP, "
          f"or they had too few overlapping months of data to correlate in Figure 13).")

    winning_vars = sorted(gdf["var_label"].dropna().unique())
    n_fallback_needed = sum(1 for v in winning_vars if v not in FIG14_FIXED_VAR_COLORS)
    # tab20 has 20 visually-distinct colours — comfortably more than the 18 indicators
    # this notebook knows about, so no two different winning variables end up sharing
    # a colour by accident.
    fallback_palette = [mcolors.to_hex(plt.get_cmap("tab20")(i)) for i in range(20)]
    var_color_map, fallback_idx = {}, 0
    for v in winning_vars:
        if v in FIG14_FIXED_VAR_COLORS:
            var_color_map[v] = FIG14_FIXED_VAR_COLORS[v]
        else:
            var_color_map[v] = fallback_palette[fallback_idx % len(fallback_palette)]
            fallback_idx += 1
    gdf["color"] = gdf["var_label"].map(var_color_map)
    gdf["color"] = gdf["color"].apply(lambda c: c if isinstance(c, (str, tuple)) else "#e6e6e6")

    fig, ax = plt.subplots(figsize=(12, 7))
    plot_gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    plot_gdf.plot(color=list(plot_gdf["color"]), edgecolor="white", linewidth=0.5, ax=ax)

    legend_elements = [Patch(facecolor=var_color_map[v], edgecolor="white", label=v) for v in winning_vars]
    ax.legend(handles=legend_elements, title="Best indicator", loc="lower left", fontsize=8, title_fontsize=8.5, framealpha=0.9)

    for _, row in plot_gdf.iterrows():
        cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
        wrapped = textwrap.fill(row["Name"], width=11, break_long_words=False, break_on_hyphens=False)
        ax.text(cx, cy, wrapped, ha="center", va="center", fontsize=9, color="#111111", alpha=0.85,
                linespacing=0.95, path_effects=[pe.withStroke(linewidth=1.8, foreground="white", alpha=0.6)])

    ax.set_title(FIG14_TITLE, fontsize=13, fontweight="bold", pad=4)
    ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "14_best_variable_per_region.png"), dpi=200, bbox_inches="tight")
    plt.show()

---
## Figure 15 — Regional correlation strength for chosen weather indicators

Side-by-side maps, one per indicator, coloured by mean Pearson correlation with media
coverage. Choose which indicators to compare below (2 or 3 is easiest to read).

In [ ]:
# ============================================================================
# FIGURE 15 — PARAMETERS
# ============================================================================

FIG15_VARS = ["soil_moisture_ufc", "soil_moisture_index", "cdi"]   # any of FIG13_DROUGHT_VARS
FIG15_VMIN, FIG15_VMAX = 0.0, 0.8    # colour-scale range for the correlation strength

# ============================================================================
# FIGURE 15 — PLOT
# ============================================================================
if not FIG13_READY:
    print("Skipping: run Figure 13's cell successfully first.")
else:
    import matplotlib.pyplot as plt
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize

    cmap = plt.cm.PuBu
    norm = Normalize(vmin=FIG15_VMIN, vmax=FIG15_VMAX)
    fig, axes = plt.subplots(1, len(FIG15_VARS), figsize=(5.3 * len(FIG15_VARS), 6.5))
    axes = np.atleast_1d(axes)

    for ax, dvar in zip(axes, FIG15_VARS):
        sub = region_corrs[region_corrs["drought_var"] == dvar]
        mean_r = sub.groupby("region_nr")["pearson_r"].mean().reset_index().rename(columns={"pearson_r": "mean_r"})
        gdf = regions_gdf.copy()
        gdf["region_nr"] = gdf["Name"].map(REGION_MAP)
        gdf = gdf.merge(mean_r, on="region_nr", how="left")
        gdf.plot(column="mean_r", cmap=cmap, norm=norm, legend=False, edgecolor="white",
                  linewidth=0.5, ax=ax, missing_kwds={"color": "#eeeeee"})
        ax.set_title(FIG13_DROUGHT_VAR_LABELS.get(dvar, dvar), fontsize=15, fontweight="bold", pad=4)
        ax.set_axis_off()

    fig.suptitle("Mean Pearson r per region", fontsize=18, fontweight="bold", y=0.95)
    fig.subplots_adjust(bottom=0.1, top=0.88, wspace=0.02)
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.30, 0.06, 0.40, 0.03])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
    cbar.set_ticks(np.linspace(FIG15_VMIN, FIG15_VMAX, 5))
    cbar.set_label("Mean Pearson r", fontsize=12)

    fig.savefig(os.path.join(FIGURES_DIR, "15_spatial_correlation.png"), dpi=200, bbox_inches="tight")
    plt.show()

---
## Figure 16 — Mean correlation by weather indicator and language

Grouped bar chart: for each indicator, the mean region-level Pearson correlation with
media coverage, one bar per language —> Do all three languages respond to the same weather signals, or different ones?

In [ ]:
FIG16_LANGUAGES = FIG13_LANGUAGES
FIG16_MAX_VARS = None      # None = all 18 indicators; set a number to show only the top N
FIG16_SORT_BY = "mean"     # "mean" = by mean r across languages, or a language code e.g. "DE"

# ============================================================================
# FIGURE 16 — PLOT
# ============================================================================
if not FIG13_READY or combined.empty:
    print("Skipping: run Figure 13's cell successfully first.")
else:
    import matplotlib.pyplot as plt

    pivot_plot = combined.pivot_table(index="var_label", columns="language", values="mean_r")
    sort_key = pivot_plot.mean(axis=1) if FIG16_SORT_BY == "mean" else pivot_plot[FIG16_SORT_BY]
    pivot_plot = pivot_plot.loc[sort_key.sort_values(ascending=False).index]
    if FIG16_MAX_VARS:
        pivot_plot = pivot_plot.head(FIG16_MAX_VARS)
    print(f"Plotting {len(pivot_plot)} indicator(s): {list(pivot_plot.index)}")

    fig, ax = plt.subplots(figsize=(max(10, 0.72 * len(pivot_plot) + 2), 5.5))
    x = np.arange(len(pivot_plot))
    width = 0.8 / max(len(FIG16_LANGUAGES), 1)
    offsets = np.linspace(-(len(FIG16_LANGUAGES) - 1) * width / 2,
                          (len(FIG16_LANGUAGES) - 1) * width / 2, len(FIG16_LANGUAGES))

    for offset, lang in zip(offsets, FIG16_LANGUAGES):
        if lang not in pivot_plot.columns:
            continue
        ax.bar(x + offset, pivot_plot[lang], width=width * 0.9, label=LANGUAGE_NAMES.get(lang, lang),
               color=LANGUAGE_COLORS.get(lang, "steelblue"))

    ax.set_xticks(x)
    ax.set_xticklabels(pivot_plot.index, rotation=45, ha="right",
                       fontsize=9 if len(pivot_plot) > 10 else 11)
    ax.set_xlim(-0.6, len(pivot_plot) - 0.4)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Mean Pearson r across regions", fontsize=11)
    ax.set_title("Mean region-level correlation by weather indicator and language",
                 fontsize=16, fontweight="bold")
    ax.legend(title="Language", fontsize=9)
    plt.tight_layout()

---
## Figure 17 — National weather indicators vs. media coverage over time

One panel per top-correlating indicator (national average), each overlaid with
normalised national media coverage. Shaded bands mark years you flag as major droughts.

In [ ]:
# ============================================================================
# FIGURE 17 — PARAMETERS
# ============================================================================

FIG17_TOP_N = 3                              # how many top indicators to show, one panel each
FIG17_LANGUAGES = FIG13_LANGUAGES
FIG17_MAJOR_DROUGHT_YEARS = (2003, 2015, 2018, 2022)   # shaded as light-grey bands

# ============================================================================
# FIGURE 17 — PLOT
# ============================================================================
if not FIG13_READY:
    print("Skipping: run Figure 13's cell successfully first.")
else:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates

    _UFC_VARS = {"soil_moisture_ufc"}
    _CDI_VARS = {"cdi", "precip_1m_index", "precip_3m_index", "precip_24m_index",
                 "hydro_index", "soil_moisture_index"}

    def display_drought_series(series, var_name):
        v = var_name.lower()
        if v in _UFC_VARS:
            return (100 - series), "Soil moisture deficit (%)\n(higher = drier)"
        if v in _CDI_VARS:
            return series - 1, f"{FIG13_DROUGHT_VAR_LABELS.get(var_name, var_name)}\n(higher = drier)"
        return -series, f"{FIG13_DROUGHT_VAR_LABELS.get(var_name, var_name)}\n(higher = drier)"

    def national_media_monthly(langs):
        media_nat = media_all_languages(langs).groupby(["year", "month", "month_start"], as_index=False)["Media"].sum()
        media_nat = media_nat.rename(columns={"Media": "Media_total"})
        baseline_combined = baseline_combined_for(langs)
        media_nat = media_nat.merge(baseline_combined[["year", "month", "month_start", "baseline_total"]],
                                     on=["year", "month", "month_start"], how="left")
        media_nat["Media_norm"] = safe_divide(media_nat["Media_total"], media_nat["baseline_total"])
        return media_nat.sort_values("month_start")

    top_vars = summary.head(FIG17_TOP_N)["drought_var"].tolist()
    drought_nat = drought_monthly.groupby(["year", "month", "month_start"], as_index=False)[top_vars].mean().sort_values("month_start")
    for v in top_vars:
        drought_nat[f"{v}_disp"], _ = display_drought_series(drought_nat[v], v)

    media_nat = national_media_monthly(FIG17_LANGUAGES)
    merged = drought_nat.merge(media_nat[["year", "month", "month_start", "Media_norm"]],
                                on=["year", "month", "month_start"], how="inner")
    merged = merged[(merged["year"] >= FIG13_START_YEAR) & (merged["year"] <= FIG13_END_YEAR)].copy()
    merged["Media_wins"] = winsorize_series(merged["Media_norm"])

    color_drought, color_media = "#1b9e77", "#d95f02"

    fig, axes = plt.subplots(FIG17_TOP_N, 1, figsize=(7.2, 3.0 * FIG17_TOP_N), sharex=True, dpi=150)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, top_vars):
        disp_series, y_label = display_drought_series(merged[var], var)
        row = summary[summary["drought_var"] == var].iloc[0]

        ax.plot(merged["month_start"], disp_series, color=color_drought, linewidth=0.9)
        ax.set_ylabel(y_label, fontsize=7.5, color=color_drought)
        ax.tick_params(axis="y", labelcolor=color_drought, labelsize=7)

        ax2 = ax.twinx()
        ax2.plot(merged["month_start"], merged["Media_wins"], color=color_media, linewidth=0.8)
        ax2.set_ylabel("Normalised media", fontsize=7.5, color=color_media)
        ax2.tick_params(axis="y", labelcolor=color_media, labelsize=7)

        for yr in FIG17_MAJOR_DROUGHT_YEARS:
            ax.axvspan(pd.Timestamp(f"{yr}-01-01"), pd.Timestamp(f"{yr + 1}-01-01"),
                       color="#bbbbbb", alpha=0.35, zorder=0, linewidth=0)

        ax.set_title(f"{FIG13_DROUGHT_VAR_LABELS.get(var, var)} (mean Pearson r = {row['mean_pearson_r']:.3f})",
                     fontsize=9.5, fontweight="bold", pad=6)
        ax.grid(axis="x", alpha=0.3, linewidth=0.6)
        ax.grid(axis="y", visible=False)
        ax2.grid(False)
        ax.xaxis.set_major_locator(mdates.YearLocator(1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.tick_params(axis="x", labelbottom=True, labelsize=7, rotation=90)

    fig.align_ylabels(axes)
    fig.suptitle("National weather indicators vs. media coverage", fontsize=13, fontweight="bold", y=0.98)
    plt.tight_layout(h_pad=2.6, rect=(0, 0, 1, 0.96))
    fig.savefig(os.path.join(FIGURES_DIR, "17_national_timeseries.png"), dpi=200, bbox_inches="tight")
    plt.show()